# Principal Portfolios with Dynamic Risk Overlays

This notebook is the clean working notebook for the repository.

**Scope.** The notebook focuses on the overlay layer: an already-investable walk-forward PCA principal portfolio is used as the base portfolio, and probabilistic volatility-state signals are translated into dynamic exposure overlays. The notebook avoids the older static-PCA/regime-diagnostics material, which belongs in the upstream PCA/regime project.

**Main retained result.** A walk-forward OU-style volatility-exceedance overlay selected on purged/embargoed development folds improves the OOS path quality of PC1 net of Corwin--Schultz implementation-cost proxies.

**Run order.** Execute cells top-to-bottom. The final validation cell builds the full comparison table, deterministic benchmarks, subperiod analysis, and stationary-bootstrap inference.


In [ ]:
# ============================================================
# 0) Imports and configuration
# ============================================================

from yfinance import download
import yfinance as yf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy.stats import norm, chi2, ncx2
from IPython.display import display

plt.rcParams["figure.figsize"] = (10, 4)
pd.options.display.float_format = "{:,.6f}".format

# ------------------------------------------------------------
# Universe and market proxies
# ------------------------------------------------------------
TICKERS_US_TECH = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "QCOM", "CSCO", "NVDA",
    "ORCL", "TXN", "ADBE", "IBM", "CRM", "AMAT", "INTU"
]

IV_TICKER_PRIMARY = "^VXN"   # Nasdaq-100 implied volatility
IV_TICKER_FALLBACK = "^VIX"  # fallback if ^VXN is unavailable
RF_TICKER_PRIMARY = "^IRX"   # 13-week Treasury bill annualized yield proxy

START = "2012-01-01"
END   = "2026-04-30"         # Yahoo end date is exclusive; adjust if rerunning later.

# Development / OOS split for overlays.
TRAIN_END_DEFAULT = "2017-12-31"
TEST_START_DEFAULT = "2018-01-01"

# PCA / walk-forward settings.
K_BAR = 7
GROSS_NORM = 1.0
WF_TRAIN_YEARS = 5
WF_STEP_MONTHS = 3
WF_K_CHOICE = 7
PC_CORE = "PC1"

# Common annualization.
ANN = 252
RF_ANN = 252

# Bootstrap settings. The selected block length is overwritten by the
# ACF-matching cell below if calibration succeeds.
BOOTSTRAP_BLOCK_LEN = 90
BOOTSTRAP_BLOCK_METHOD = "fallback fixed value"

# Reproducibility for bootstrap-style simulation steps.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# ============================================================
# 1) Reusable helpers
# ============================================================

def linear_returns_from_prices(adj_close: pd.DataFrame | pd.Series) -> pd.DataFrame | pd.Series:
    """Simple returns from adjusted close prices."""
    return adj_close.pct_change().dropna(how="all")


def annualized_yield_pct_to_daily_return(y_ann_pct: pd.Series, ann: int = 252) -> pd.Series:
    """Convert annualized yield in percent to daily simple return."""
    y = pd.Series(y_ann_pct).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    y = (y / 100.0).clip(lower=-0.999999)
    out = (1.0 + y) ** (1.0 / ann) - 1.0
    out.name = "rf_daily"
    return out


def weighted_mean_cov(X: pd.DataFrame):
    """Equal-weight sample mean/covariance matrix."""
    mu = X.mean(axis=0).values
    Xc = X.values - mu.reshape(1, -1)
    Sigma = (Xc.T @ Xc) / X.shape[0]
    return (
        pd.Series(mu, index=X.columns, name="mu"),
        pd.DataFrame(Sigma, index=X.columns, columns=X.columns),
    )


def correlation_pca(Sigma: np.ndarray):
    """PCA of the correlation matrix implied by covariance Sigma."""
    Sigma = np.asarray(Sigma, dtype=float)
    vol = np.sqrt(np.clip(np.diag(Sigma), 1e-18, None))
    Dinv = np.diag(1.0 / vol)
    rho = Dinv @ Sigma @ Dinv
    lam, V = np.linalg.eigh(rho)
    lam = lam[::-1]
    V = V[:, ::-1]

    # Deterministic sign convention.
    max_abs_row = np.argmax(np.abs(V), axis=0)
    s = np.sign(V[max_abs_row, np.arange(V.shape[0])])
    s[s == 0] = 1.0
    V = V * s.reshape(1, -1)
    return lam, V, vol, rho


def pc_portfolio_weights_from_corr_pca(
    V: np.ndarray,
    vol: np.ndarray,
    gross_norm: float = 1.0,
) -> pd.DataFrame:
    """Factor-mimicking principal-portfolio weights with unit gross exposure."""
    V = np.asarray(V, dtype=float)
    vol = np.asarray(vol, dtype=float)
    W = V * (1.0 / vol).reshape(-1, 1)
    gross = np.sum(np.abs(W), axis=0)
    gross[gross == 0] = 1.0
    W = W / gross.reshape(1, -1) * gross_norm
    return pd.DataFrame(W)


def factor_returns(R: pd.DataFrame, W: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        R.values @ W.values,
        index=R.index,
        columns=[f"PC{i+1}" for i in range(W.shape[1])],
    )


def perf_stats(r: pd.Series, ann: int = ANN) -> dict:
    """CAGR / volatility / Sharpe / Sortino / MaxDD / Calmar / Martin."""
    r = pd.Series(r).dropna().astype(float)
    if len(r) < 5:
        return {"n": len(r), "CAGR": np.nan, "AnnVol": np.nan, "Sharpe": np.nan,
                "Sortino": np.nan, "MaxDD": np.nan, "Calmar": np.nan, "Martin": np.nan}

    wealth = (1.0 + r).cumprod()
    n = len(r)
    years = n / ann
    cagr = wealth.iloc[-1] ** (1.0 / years) - 1.0 if years > 0 else np.nan
    ann_vol = r.std(ddof=1) * np.sqrt(ann)
    ann_ret = r.mean() * ann
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan

    downside = r[r < 0]
    down_vol = downside.std(ddof=1) * np.sqrt(ann) if len(downside) > 1 else np.nan
    sortino = ann_ret / down_vol if pd.notna(down_vol) and down_vol > 0 else np.nan

    dd = wealth / wealth.cummax() - 1.0
    max_dd = dd.min()
    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan
    ulcer = np.sqrt(np.mean((100.0 * dd) ** 2)) / 100.0
    martin = cagr / ulcer if ulcer > 0 else np.nan

    return {
        "n": n,
        "CAGR": cagr,
        "AnnVol": ann_vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "MaxDD": max_dd,
        "Calmar": calmar,
        "Martin": martin,
    }


def perf_table(return_dict: dict[str, pd.Series], ann: int = ANN) -> pd.DataFrame:
    rows = [{"strategy": name, **perf_stats(r, ann=ann)} for name, r in return_dict.items()]
    return pd.DataFrame(rows).set_index("strategy")


def rolling_realized_vol(
    r: pd.Series,
    window: int = 21,
    ann: int = 252,
    min_obs: int | None = None,
) -> pd.Series:
    r = pd.Series(r).astype(float)
    if min_obs is None:
        min_obs = max(5, window // 2)
    return (r.rolling(window, min_periods=min_obs).std(ddof=1) * np.sqrt(ann)).rename(f"rv_{window}")


def rolling_downside_semivol(
    r: pd.Series,
    window: int = 21,
    ann: int = 252,
    mar: float = 0.0,
) -> pd.Series:
    r = pd.Series(r).astype(float)
    neg = np.minimum(r - mar, 0.0)
    return np.sqrt(
        ann * pd.Series(neg**2, index=r.index).rolling(
            window, min_periods=max(5, window // 2)
        ).mean()
    ).rename(f"dsv_{window}")


def forward_realized_vol_from_returns(r: pd.Series, horizon: int = 21, ann: int = 252) -> pd.Series:
    r = pd.Series(r).dropna().astype(float)
    x = r.values
    out = np.full(len(x), np.nan)
    for i in range(len(x)):
        j = min(len(x), i + horizon + 1)
        fwd = x[i + 1:j]
        if len(fwd) < max(5, horizon // 2):
            continue
        out[i] = np.std(fwd, ddof=1) * np.sqrt(ann)
    return pd.Series(out, index=r.index, name=f"rv_fwd_{horizon}")


def auc_rank(y_true: pd.Series, p_hat: pd.Series) -> float:
    """Rank-based AUC without sklearn dependency."""
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return np.nan
    y = df["y"].astype(int).values
    p = df["p"].astype(float).values
    ranks = pd.Series(p).rank(method="average").values
    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.nan
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def brier_score(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty:
        return np.nan
    return float(np.mean((df["p"].astype(float) - df["y"].astype(float)) ** 2))


def stationary_bootstrap_indices(T: int, avg_block_len: float, rng: np.random.Generator) -> np.ndarray:
    """Politis--Romano stationary-bootstrap index generator."""
    p = 1.0 / avg_block_len
    idx = np.empty(T, dtype=int)
    idx[0] = rng.integers(0, T)
    for t in range(1, T):
        if rng.random() < p:
            idx[t] = rng.integers(0, T)
        else:
            idx[t] = (idx[t - 1] + 1) % T
    return idx


def _acf(x, nlags: int = 30) -> np.ndarray:
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    if len(x) < nlags + 5:
        return np.full(nlags + 1, np.nan)
    x = x - x.mean()
    denom = np.dot(x, x)
    if denom <= 0:
        return np.full(nlags + 1, np.nan)

    out = [1.0]
    for k in range(1, nlags + 1):
        out.append(float(np.dot(x[:-k], x[k:]) / denom))
    return np.asarray(out)


def choose_block_length_by_acf_matching(
    s: pd.Series,
    candidates: list[int],
    nlags: int = 30,
    n_boot: int = 300,
    use_abs: bool = True,
    seed: int = 1,
    stationary_bootstrap_indices_fn=stationary_bootstrap_indices,
    distance: str = "weighted",
) -> dict:
    """Choose mean block length by matching the ACF of returns or absolute returns."""
    r = pd.Series(s).dropna().astype(float)
    x = np.abs(r.values) if use_abs else r.values
    T = len(x)
    if T < nlags + 10:
        raise ValueError("Series too short for requested nlags.")

    target = _acf(x, nlags=nlags)
    rng = np.random.default_rng(seed)
    rows = []
    boot_means = {}

    for L in candidates:
        acfs = np.zeros((n_boot, nlags + 1))
        for b in range(n_boot):
            ii = stationary_bootstrap_indices_fn(T, float(L), rng)
            acfs[b, :] = _acf(x[ii], nlags=nlags)
        acf_mean = np.nanmean(acfs, axis=0)
        boot_means[L] = acf_mean

        diff = acf_mean[1:] - target[1:]
        if distance == "weighted":
            w = 1.0 / np.arange(1, nlags + 1)
            score = float(np.sqrt(np.nanmean((diff**2) * w)))
        elif distance == "l1":
            score = float(np.nanmean(np.abs(diff)))
        elif distance == "l2":
            score = float(np.sqrt(np.nanmean(diff**2)))
        else:
            raise ValueError("distance must be 'weighted', 'l1', or 'l2'.")

        rows.append({"block_len": int(L), "acf_distance": score})

    scores = pd.DataFrame(rows).sort_values("acf_distance").reset_index(drop=True)
    return {
        "best_L": int(scores.iloc[0]["block_len"]),
        "scores": scores,
        "target_acf": target,
        "boot_acf_mean": boot_means,
        "use_abs": use_abs,
        "nlags": nlags,
        "n_boot": n_boot,
        "distance": distance,
    }


def politis_white_block_length(x, use_abs: bool = True) -> int:
    """
    Lightweight diagnostic inspired by Politis--White logic.
    Used as a secondary reference only; not the exact original estimator.
    """
    s = pd.Series(x).dropna().astype(float)
    if use_abs:
        s = s.abs()
    ac = _acf(s, nlags=min(60, max(5, len(s) // 10)))
    pos = np.where(ac[1:] <= 0)[0]
    first_zero = int(pos[0] + 1) if len(pos) else min(43, len(ac) - 1)
    return max(5, min(90, first_zero * 3))


def add_months(ts: pd.Timestamp, m: int) -> pd.Timestamp:
    return (pd.Timestamp(ts) + pd.offsets.DateOffset(months=m)).normalize()


def _align_W_to_prev(W_prev: pd.DataFrame | None, W_now: pd.DataFrame) -> pd.DataFrame:
    """Align PC signs to previous rebalance so sign flips do not create fake turnover/costs."""
    if W_prev is None:
        return W_now.copy()

    W_aligned = W_now.copy()
    common_cols = [c for c in W_now.columns if c in W_prev.columns]
    for c in common_cols:
        if np.dot(W_prev[c].values.astype(float), W_now[c].values.astype(float)) < 0:
            W_aligned[c] = -W_aligned[c]
    return W_aligned


def walk_forward_pca(
    R: pd.DataFrame,
    start_test: str,
    train_years: int = 5,
    step_months: int = 3,
    k: int = 7,
    gross_norm: float = 1.0,
    tc_bps: float | None = 0.0,
    asset_costs: pd.DataFrame | None = None,
    charge_initial_rebalance: bool = False,
):
    """
    Walk-forward PCA with sign alignment and rebalance costs.
    If asset_costs is provided, costs are charged asset-by-asset using one-way costs.
    """
    R = R.dropna().sort_index().copy()
    t0 = pd.Timestamp(start_test)
    t_end = R.index.max()

    if asset_costs is not None:
        asset_costs = (
            asset_costs.reindex(index=R.index, columns=R.columns)
            .sort_index()
            .ffill()
            .fillna(0.0)
            .astype(float)
        )

    weights_hist, f_all, costs_all = [], [], []
    W_prev = None
    reb = t0

    while True:
        train_start = reb - pd.DateOffset(years=train_years)
        train_end = reb - pd.DateOffset(days=1)
        test_start = reb
        test_end = add_months(reb, step_months) - pd.DateOffset(days=1)

        if test_start > t_end:
            break

        R_train = R.loc[train_start:train_end]
        R_test = R.loc[test_start:test_end]

        if len(R_train) < 252 or len(R_test) < 5:
            reb = add_months(reb, step_months)
            continue

        _, Sigma = weighted_mean_cov(R_train)
        _, V, vol, _ = correlation_pca(Sigma.values)

        W = pc_portfolio_weights_from_corr_pca(V[:, :k], vol, gross_norm=gross_norm)
        W.index = R.columns
        W.columns = [f"PC{i+1}" for i in range(W.shape[1])]
        W = _align_W_to_prev(W_prev, W)

        f = factor_returns(R_test, W)
        f_all.append(f)

        seg_cost = pd.DataFrame(0.0, index=R_test.index, columns=f.columns)
        if charge_initial_rebalance or (W_prev is not None):
            dW = W.abs() if W_prev is None else (W - W_prev).abs()
            trade_date = R_test.index[0]

            if asset_costs is not None:
                c_assets = asset_costs.loc[trade_date, R.columns]
                c_pc = dW.mul(c_assets, axis=0).sum(axis=0)
            else:
                c = 0.0 if tc_bps is None else float(tc_bps) / 1e4
                c_pc = dW.sum(axis=0) * c

            seg_cost.iloc[0, :] = c_pc.reindex(f.columns).values

        costs_all.append(seg_cost)
        weights_hist.append({
            "rebalance": reb,
            "train_start": train_start,
            "train_end": train_end,
            "W": W.copy(),
        })

        W_prev = W.copy()
        reb = add_months(reb, step_months)

    F = pd.concat(f_all).sort_index() if f_all else pd.DataFrame()
    C = pd.concat(costs_all).sort_index() if costs_all else pd.DataFrame(0.0, index=F.index, columns=F.columns)
    C = C.reindex(index=F.index, columns=F.columns).fillna(0.0)
    return F, F - C, weights_hist


## Current reference results

Reference run used in the SSRN working-paper draft: **OOS Jan 2018--Apr 2026**, aligned sample ending **2026-04-14**.

| Strategy | CAGR | AnnVol | Sharpe | MaxDD | Martin |
|---|---:|---:|---:|---:|---:|
| PC1 base | 20.74% | 25.25% | 0.873 | -35.85% | 1.868 |
| Retained OU overlay, net CS | 18.73% | 19.56% | 0.976 | -26.41% | 2.213 |
| Combined calibrated, net CS | 18.83% | 20.24% | 0.954 | -28.81% | 2.087 |
| Vol target 20%, net CS | 17.37% | 18.68% | 0.951 | -26.53% | 1.943 |
| QQQ | 19.18% | 23.89% | 0.854 | -35.12% | 1.689 |

Bootstrap evidence for the retained OU overlay versus PC1 base (`N=2000`, mean block length 90 trading days, ACF-matching on absolute QQQ returns):

| Metric | Mean delta | 5% | Median | 95% | P(target wins) |
|---|---:|---:|---:|---:|---:|
| Delta Sharpe | 0.0950 | -0.0348 | 0.0955 | 0.2190 | 89.35% |
| Delta MaxDD | 0.0745 | 0.0053 | 0.0768 | 0.1383 | 98.30% |

**Interpretation.** The retained OU layer is a path-quality overlay: it reduces volatility and drawdown materially, while giving up some upside participation. Medium/Fast/Combined layers are retained as challengers and research extensions, not as the current headline result.


## 1) Data download and base panels

In [ ]:
all_tickers = TICKERS_US_TECH + [IV_TICKER_PRIMARY, IV_TICKER_FALLBACK]
px = download(all_tickers, start=START, end=END, auto_adjust=False, progress=False).dropna(how="all")

px_adj_close_all = px["Adj Close"][TICKERS_US_TECH].copy()
px_high = px["High"][TICKERS_US_TECH].copy()
px_low = px["Low"][TICKERS_US_TECH].copy()

# For PCA we still want a common-support panel.
px_stocks = px_adj_close_all.dropna(how="any")

px_adj = px["Adj Close"].copy()
iv = px_adj[IV_TICKER_PRIMARY] if IV_TICKER_PRIMARY in px_adj.columns else None
if iv is None or iv.dropna().empty:
    iv = px_adj[IV_TICKER_FALLBACK]
iv.name = "IV"

R = linear_returns_from_prices(px_stocks)

# -------------------------
# Historical risk-free proxy
# -------------------------
rf_raw = download(RF_TICKER_PRIMARY, start=START, end=END, auto_adjust=False, progress=False)

def _extract_single_close(x):
    if isinstance(x, pd.Series):
        return x.astype(float)
    if not isinstance(x, pd.DataFrame):
        return pd.Series(dtype=float)
    for field in ["Adj Close", "Close"]:
        if field in x.columns:
            s = x[field]
            if isinstance(s, pd.DataFrame):
                s = s.iloc[:, 0]
            return s.astype(float)
    if x.shape[1] == 1:
        return x.iloc[:, 0].astype(float)
    return pd.Series(dtype=float)

rf_ann_pct = _extract_single_close(rf_raw).dropna()
if rf_ann_pct.empty:
    print(f"WARNING: no historical risk-free proxy downloaded from {RF_TICKER_PRIMARY}; using zero daily RF.")
    rf_daily_hist = pd.Series(0.0, index=R.index, name="rf_daily")
    rf_source_used = "ZERO_FALLBACK"
else:
    rf_daily_hist = annualized_yield_pct_to_daily_return(rf_ann_pct, ann=RF_ANN)
    rf_daily_hist = rf_daily_hist.reindex(R.index).ffill().fillna(0.0)
    rf_source_used = RF_TICKER_PRIMARY

print("R shape:", R.shape)
print(
    f"RF source: {rf_source_used} | "
    f"non-null daily obs on R index: {int(rf_daily_hist.notna().sum())} | "
    f"sample mean daily RF (bps): {1e4 * rf_daily_hist.mean():.4f}"
)
R.head()


## 2) Bootstrap block-length calibration

In [ ]:
# ============================================================
# Bootstrap block-length calibration (QQQ benchmark)
# ============================================================

px_B = download("QQQ", start=START, end=END, auto_adjust=False, progress=False)["Adj Close"].dropna(how="all")
B_ret = linear_returns_from_prices(px_B)

if isinstance(B_ret, pd.DataFrame):
    B_ret = B_ret.iloc[:, 0]
elif isinstance(B_ret, np.ndarray) and B_ret.ndim == 2 and B_ret.shape[1] == 1:
    B_ret = B_ret[:, 0]
B_ret = pd.Series(B_ret).dropna().astype(float)
B_ret.name = "QQQ"

# We use ACF-matching on |QQQ returns| as the primary selector because the
# combined-overlay bootstrap is especially sensitive to volatility clustering
# and drawdown-path dependence.  The Politis--White-style value is kept as a
# diagnostic, but the implementation below is a lightweight heuristic, not the
# exact original estimator.
BLOCK_CANDIDATES = [5, 10, 15, 20, 30, 43, 60, 90]
res_abs = choose_block_length_by_acf_matching(
    B_ret.abs(),
    candidates=BLOCK_CANDIDATES,
    nlags=30,
    n_boot=300,
    use_abs=True,
    seed=1,
    stationary_bootstrap_indices_fn=stationary_bootstrap_indices,
    distance="weighted",
)

BOOTSTRAP_BLOCK_LEN = int(res_abs["best_L"])
BOOTSTRAP_BLOCK_METHOD = "ACF-matching on |QQQ returns|"

print("ACF-matching best L on |QQQ returns|:", BOOTSTRAP_BLOCK_LEN)
display(res_abs["scores"])

L_pw = politis_white_block_length(B_ret, use_abs=True)
print("Politis--White-style suggested mean block length:", L_pw)
print(f"Selected bootstrap mean block length for downstream validation: {BOOTSTRAP_BLOCK_LEN} ({BOOTSTRAP_BLOCK_METHOD})")


## 3) Corwin--Schultz costs and walk-forward PC1 design panel

In [ ]:
# ============================================================
# 4) Corwin--Schultz costs and walk-forward PC1 panel
# ============================================================

def corwin_schultz_spread(high: pd.Series, low: pd.Series) -> pd.Series:
    """
    Corwin--Schultz (2012) bid-ask spread estimator from daily high/low prices.
    Returns the FULL proportional spread, e.g. 0.01 = 1%.
    """
    high = pd.Series(high).astype(float)
    low = pd.Series(low).astype(float)
    idx = high.dropna().index.intersection(low.dropna().index)

    high = high.loc[idx].sort_index()
    low = low.loc[idx].sort_index()

    hl = np.log(high / low)
    beta = hl.pow(2) + hl.shift(1).pow(2)

    high2 = pd.concat([high, high.shift(1)], axis=1).max(axis=1)
    low2 = pd.concat([low, low.shift(1)], axis=1).min(axis=1)
    gamma = np.log(high2 / low2).pow(2)

    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(gamma / k)
    alpha = alpha.clip(lower=0)

    spread = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    spread.name = "cs_spread"
    return spread


def corwin_schultz_panel(
    px_high: pd.DataFrame,
    px_low: pd.DataFrame,
    tickers: list | None = None,
    clip_upper: float | None = None,
) -> pd.DataFrame:
    if tickers is None:
        tickers = [c for c in px_high.columns if c in px_low.columns]

    out = {}
    for t in tickers:
        h = px_high[t].dropna()
        l = px_low[t].dropna()
        idx = h.index.intersection(l.index)

        if len(idx) < 3:
            out[t] = pd.Series(dtype=float)
            continue

        s = corwin_schultz_spread(h.loc[idx], l.loc[idx])
        if clip_upper is not None:
            s = s.clip(upper=clip_upper)
        out[t] = s

    return pd.DataFrame(out).sort_index().reindex(columns=tickers)


def cs_panel_to_one_way_cost(cs_spreads: pd.DataFrame) -> pd.DataFrame:
    """Convert full spread to one-way implementation cost."""
    return 0.5 * cs_spreads.astype(float)


# ------------------------------------------------------------
# Asset-level one-way cost panel
# ------------------------------------------------------------
cs_spreads = corwin_schultz_panel(
    px_high=px_high,
    px_low=px_low,
    tickers=list(R.columns),
    clip_upper=None,
)

asset_costs = (
    cs_panel_to_one_way_cost(cs_spreads)
    .reindex(index=R.index, columns=R.columns)
    .ffill()
    .fillna(0.0)
)

print("One-way Corwin--Schultz cost summary (bps) by asset:")
display((1e4 * asset_costs).describe(percentiles=[0.50, 0.95]).T[["mean", "50%", "95%", "max"]].sort_values("50%"))


# ------------------------------------------------------------
# Walk-forward PC1 base portfolio
# ------------------------------------------------------------
DEV_END = pd.Timestamp(TRAIN_END_DEFAULT)
OOS_START = pd.Timestamp(TEST_START_DEFAULT)
WF_FIRST_TRADABLE = (pd.Timestamp(START) + pd.DateOffset(years=WF_TRAIN_YEARS)).normalize()

print("Earliest feasible walk-forward start:", WF_FIRST_TRADABLE.date())
print("Development end:", DEV_END.date(), "| OOS start:", OOS_START.date())

F_wf_long, F_wf_net_long, W_hist_long = walk_forward_pca(
    R=R,
    start_test=str(WF_FIRST_TRADABLE.date()),
    train_years=WF_TRAIN_YEARS,
    step_months=WF_STEP_MONTHS,
    k=WF_K_CHOICE,
    gross_norm=GROSS_NORM,
    tc_bps=None,
    asset_costs=asset_costs,
    charge_initial_rebalance=False,
)

pc1_long = F_wf_net_long[PC_CORE].dropna().astype(float).copy()
pc1_long.name = "pc1_base"

H_MEDIUM = 21
BASE_COL = "pc1_base"

panel = pd.DataFrame({"pc1_base": pc1_long})

# Realized-volatility features for HAR / OU layers.
panel["rv_5"] = rolling_realized_vol(panel["pc1_base"], window=5, ann=ANN)
panel["rv_21"] = rolling_realized_vol(panel["pc1_base"], window=21, ann=ANN)
panel["rv_63"] = rolling_realized_vol(panel["pc1_base"], window=63, ann=ANN)
panel["dsv_21"] = rolling_downside_semivol(panel["pc1_base"], window=21, ann=ANN)
panel["rv_fwd_H"] = forward_realized_vol_from_returns(panel["pc1_base"], horizon=H_MEDIUM, ann=ANN)

# HAR lagged components.
panel["rv_21_lag1"] = panel["rv_21"].shift(1)
panel["rv_5_lag1"] = panel["rv_5"].shift(1)
panel["rv_21_lag5_mean"] = panel["rv_21"].shift(1).rolling(5, min_periods=5).mean()
panel["rv_21_lag21_mean"] = panel["rv_21"].shift(1).rolling(21, min_periods=15).mean()

panel = panel.dropna().copy()
dev = panel.loc[:DEV_END].copy()
oos = panel.loc[OOS_START:].copy()

print("Long walk-forward PC1:", pc1_long.index.min().date(), "->", pc1_long.index.max().date(), "| n =", len(pc1_long))
print("Panel:", panel.index.min().date(), "->", panel.index.max().date(), "| n =", len(panel))
print("DEV:", dev.index.min().date() if len(dev) else None, "->", dev.index.max().date() if len(dev) else None, "| n =", len(dev))
print("OOS:", oos.index.min().date() if len(oos) else None, "->", oos.index.max().date() if len(oos) else None, "| n =", len(oos))

print("\nBase PC1 OOS performance:")
display(perf_table({"PC1_base": oos["pc1_base"]}).round(4))
display(panel.head())


## 4) Medium-horizon HAR-RV residual-bootstrap challenger

In [ ]:
# ============================================================
# V5 — HAR-RV + residual bootstrap predictive distribution
# Goal: estimate P( RV_fwd_H > tau | information at t )
# using HAR mean dynamics + bootstrapped residual uncertainty
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

# ------------------------------------------------------------
# Preconditions:
# expects dev / oos from V3.4 already available
# with columns:
#   rv_fwd_H, rv_5_lag1, rv_21_lag5_mean, rv_21_lag21_mean
# ------------------------------------------------------------
assert "dev" in globals() and "oos" in globals(), "Run the base-panel cell first so that dev and oos exist."

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
BOOT_B = 5000                 # number of bootstrap draws
TAU_MODE = "dev_percentile"   # "dev_percentile" or "absolute"
TAU_PERCENTILE = 0.80         # e.g. 0.80, 0.85, 0.90
TAU_ABS = 0.30                # used only if TAU_MODE == "absolute"

HAR_COLS = ["rv_5_lag1", "rv_21_lag5_mean", "rv_21_lag21_mean"]
Y_COL = "rv_fwd_H"

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def auc_rank(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return np.nan
    y = df["y"].astype(int).values
    p = df["p"].astype(float).values
    ranks = pd.Series(p).rank(method="average").values
    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.nan
    auc = (ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0)
    return float(auc)

def brier_score(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty:
        return np.nan
    return float(np.mean((df["p"].astype(float) - df["y"].astype(float)) ** 2))

def bucket_prob_calibration(y_true: pd.Series, p_hat: pd.Series, q: int = 5) -> pd.DataFrame:
    tmp = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna().copy()
    if tmp.empty:
        return pd.DataFrame()
    tmp["bucket"] = pd.qcut(tmp["p"], q=q, duplicates="drop")
    out = tmp.groupby("bucket", observed=False).agg(
        n=("y", "size"),
        mean_p=("p", "mean"),
        realized_rate=("y", "mean"),
    )
    out["realized_rate"] *= 100
    out["mean_p"] *= 100
    return out

# ------------------------------------------------------------
# 1) Fit HAR on development sample
# ------------------------------------------------------------
har_dev = dev[[Y_COL] + HAR_COLS].dropna().copy()
X_dev = sm.add_constant(har_dev[HAR_COLS], has_constant="add")
y_dev = har_dev[Y_COL].astype(float)

har_res = sm.OLS(y_dev, X_dev).fit()
har_resid = (y_dev - har_res.fittedvalues).dropna()

print("HAR fit on development sample")
print("n_dev:", len(har_dev))
display(pd.DataFrame({
    "coef": har_res.params,
    "tstat": har_res.tvalues,
    "pvalue": har_res.pvalues,
}))

# ------------------------------------------------------------
# 2) Choose threshold tau
# ------------------------------------------------------------
if TAU_MODE == "dev_percentile":
    tau = float(dev[Y_COL].dropna().quantile(TAU_PERCENTILE))
    tau_label = f"dev p{int(TAU_PERCENTILE*100)}"
else:
    tau = float(TAU_ABS)
    tau_label = f"absolute {TAU_ABS:.4f}"

print(f"\nChosen volatility threshold tau = {tau:.6f} ({tau_label})")

# ------------------------------------------------------------
# 3) OOS predictive distribution via residual bootstrap
#    Frozen-coefficient version first (clean baseline)
# ------------------------------------------------------------
har_oos = oos[[Y_COL] + HAR_COLS].dropna().copy()
X_oos = sm.add_constant(har_oos[HAR_COLS], has_constant="add")
mu_oos = pd.Series(har_res.predict(X_oos), index=har_oos.index, name="mu_hat")

# bootstrap residual draws
resid_pool = har_resid.values.astype(float)
boot_draws = np.random.choice(resid_pool, size=(len(mu_oos), BOOT_B), replace=True)

# predictive draws: mean + resampled residual
pred_draws = mu_oos.values.reshape(-1, 1) + boot_draws

# optional clipping to nonnegative values
pred_draws = np.clip(pred_draws, 1e-8, None)

# predictive summaries
p_exceed = pd.Series((pred_draws > tau).mean(axis=1), index=mu_oos.index, name="p_rv_exceed")
q90 = pd.Series(np.quantile(pred_draws, 0.90, axis=1), index=mu_oos.index, name="rv_q90")
pred_mean = pd.Series(pred_draws.mean(axis=1), index=mu_oos.index, name="rv_pred_mean")
pred_std = pd.Series(pred_draws.std(axis=1, ddof=1), index=mu_oos.index, name="rv_pred_std")

# realized event
event_oos = (har_oos[Y_COL] > tau).astype(int).rename("event_rv_exceed")

# ------------------------------------------------------------
# 4) Diagnostics
# ------------------------------------------------------------
diag = pd.DataFrame({
    "realized_rv_fwd": har_oos[Y_COL],
    "rv_pred_mean": pred_mean,
    "rv_pred_std": pred_std,
    "rv_q90": q90,
    "p_rv_exceed": p_exceed,
    "event_rv_exceed": event_oos,
})

print("\nOOS predictive distribution diagnostics")
summary_diag = pd.DataFrame([{
    "oos_n": len(diag),
    "tau": tau,
    "event_rate_%": 100 * diag["event_rv_exceed"].mean(),
    "mean_pred_p_%": 100 * diag["p_rv_exceed"].mean(),
    "AUC": auc_rank(diag["event_rv_exceed"], diag["p_rv_exceed"]),
    "Brier": brier_score(diag["event_rv_exceed"], diag["p_rv_exceed"]),
}]).T
summary_diag.columns = ["value"]
display(summary_diag)

print("\nProbability calibration by buckets")
display(bucket_prob_calibration(diag["event_rv_exceed"], diag["p_rv_exceed"], q=5))

print("\nTail of OOS predictive table")
display(diag.tail(10))

# ------------------------------------------------------------
# 5) Simple plots
# ------------------------------------------------------------
ax = diag[["realized_rv_fwd", "rv_pred_mean", "rv_q90"]].plot(
    title=f"V5 — OOS HAR predictive distribution | tau={tau:.3f}"
)
ax.set_ylabel("forward realized volatility")

ax2 = diag["p_rv_exceed"].plot(
    title=f"V5 — OOS probability that forward RV exceeds tau={tau:.3f}"
)
ax2.set_ylabel("probability")

# ------------------------------------------------------------
# 6) Save useful objects for next overlay step
# ------------------------------------------------------------
har_boot_oos = diag.copy()
har_boot_oos["tau"] = tau
har_boot_oos["tau_mode"] = tau_label

print("\nSaved object: har_boot_oos")


## 5) Retained slow layer: OU volatility-exceedance model and robustness checks

In [ ]:
# ============================================================
# V6b — OU slow-layer design sweep *(selection fixed)*
# Purpose: horizon / state-variable / threshold sensitivity.
#
# Important correction vs the earlier version:
#   - candidates are NOT selected by OOS Sharpe;
#   - the preferred candidate is selected by DEVELOPMENT AUC of the OU exceedance score;
#   - development overlay performance is reported, but not used when dev_trades=0;
#   - OOS rankings are diagnostics/robustness only.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm
from IPython.display import display
import matplotlib.pyplot as plt

assert "dev" in globals() and "oos" in globals(), "Run the base-panel cell first."
assert "har_boot_oos" in globals(), "Run V5 first so that the retained HAR path is already available."

BASE_COL = "pc1_base"
ANN = 252

# For the sweep only, use a lighter minimum calibration window so that the
# 2017 development sample can actually rank candidates without looking at OOS.
# The main V6 OU layer remains the pre-specified benchmark layer.
MIN_CALIB_OBS = 126
MIN_SELECTION_SCORE_OBS = 30

# Same mild weekly hysteresis as the original slow challenger
SWEEP_ENTER_DEF_Q = 0.85
SWEEP_EXIT_DEF_Q  = 0.70
SWEEP_ENTER_EXT_Q = 0.96
SWEEP_EXIT_EXT_Q  = 0.85
SWEEP_EXP_NORMAL  = 1.00
SWEEP_EXP_DEF     = 0.75
SWEEP_EXP_EXT     = 0.50
MIN_HIST_WEEKS    = 10

HORIZON_GRID = [21, 42, 63]
TAU_GRID = [0.80, 0.85]


def _rolling_rv_sweep(r, window=21, ann=252):
    return (
        pd.Series(r).astype(float)
        .rolling(window, min_periods=max(5, window // 2))
        .std(ddof=1) * np.sqrt(ann)
    ).rename(f"rv_{window}")


def _rolling_dsv_sweep(r, window=21, ann=252, mar=0.0):
    r = pd.Series(r).astype(float)
    neg = np.minimum(r - mar, 0.0)
    return np.sqrt(
        ann * pd.Series(neg**2, index=r.index).rolling(window, min_periods=max(5, window // 2)).mean()
    ).rename(f"dsv_{window}")


def fit_ou_positive_series(x):
    x = pd.Series(x).dropna().astype(float)
    x = x[x > 0]
    if len(x) < 30:
        return None
    logx = np.log(x)
    y = logx.iloc[1:].values
    X = sm.add_constant(logx.iloc[:-1].values, has_constant="add")
    try:
        res = sm.OLS(y, X).fit()
    except Exception:
        return None
    alpha, beta = float(res.params[0]), float(res.params[1])
    xi = float(np.std(res.resid, ddof=2))
    if not np.isfinite(beta) or beta <= 0 or beta >= 1:
        return None
    kappa = -np.log(beta)
    theta = alpha / (1.0 - beta)
    return {"alpha": alpha, "beta": beta, "xi": xi, "kappa": kappa, "theta": theta}


def ou_predictive(log_x_t, params, H):
    kappa, theta, xi = params["kappa"], params["theta"], params["xi"]
    exp_kH = np.exp(-kappa * H)
    mu_H   = theta + (log_x_t - theta) * exp_kH
    var_H  = (xi ** 2) / (2.0 * kappa) * (1.0 - np.exp(-2.0 * kappa * H))
    std_H  = np.sqrt(max(var_H, 1e-12))
    return mu_H, std_H


def auc_rank(y_true, p_hat):
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return np.nan
    y = df["y"].astype(int).values
    p = df["p"].astype(float).values
    ranks = pd.Series(p).rank(method="average").values
    n1 = int(y.sum()); n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.nan
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def brier_score(y_true, p_hat):
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty:
        return np.nan
    return float(np.mean((df["p"].astype(float) - df["y"].astype(float)) ** 2))


def _perf(r, ann=252):
    r = pd.Series(r).dropna().astype(float)
    if len(r) < 5:
        return {"n": len(r), "CAGR": np.nan, "AnnVol": np.nan, "Sharpe": np.nan,
                "Sortino": np.nan, "MaxDD": np.nan, "Calmar": np.nan, "Martin": np.nan}
    w = (1 + r).cumprod()
    n = len(r); yrs = n / ann
    cagr = w.iloc[-1]**(1/yrs)-1 if yrs > 0 else np.nan
    vol = r.std(ddof=1) * np.sqrt(ann)
    mu = r.mean() * ann
    shr = mu / vol if vol > 0 else np.nan
    dn = r[r < 0]
    dvol = dn.std(ddof=1) * np.sqrt(ann) if len(dn) > 1 else np.nan
    srt = mu / dvol if pd.notna(dvol) and dvol > 0 else np.nan
    pk = w.cummax(); dd = w / pk - 1; mdd = dd.min()
    cal = cagr / abs(mdd) if mdd < 0 else np.nan
    ulc = np.sqrt(np.mean((100 * dd) ** 2)) / 100
    mar = cagr / ulc if ulc > 0 else np.nan
    return {"n": n, "CAGR": cagr, "AnnVol": vol, "Sharpe": shr, "Sortino": srt,
            "MaxDD": mdd, "Calmar": cal, "Martin": mar}


def make_ou_scores(eval_index, state_full, H, tau, name):
    scores = pd.Series(np.nan, index=eval_index, name=name)
    for i, dt in enumerate(eval_index):
        hist = state_full.loc[:dt].dropna()
        if len(hist) < MIN_CALIB_OBS:
            continue
        params = fit_ou_positive_series(hist)
        if params is None:
            continue
        x_now = float(hist.iloc[-1])
        mu_H, std_H = ou_predictive(np.log(x_now), params, H=H)
        z = (np.log(tau) - mu_H) / std_H
        scores.loc[dt] = float(1.0 - norm.cdf(z))
    return scores


def build_weekly_overlay_from_scores(scores, base_r):
    base_r = pd.Series(base_r).dropna().astype(float)
    score_weekly = pd.Series(scores).dropna().resample("W-FRI").last().dropna()
    state = "normal"
    rows = []
    for i, dt in enumerate(score_weekly.index):
        p_now = float(score_weekly.loc[dt])
        hist = score_weekly.iloc[:i]
        if len(hist) < MIN_HIST_WEEKS or np.isnan(p_now):
            state = "normal"
            exposure = SWEEP_EXP_NORMAL
        else:
            q_de = float(hist.quantile(SWEEP_ENTER_DEF_Q))
            q_dx = float(hist.quantile(SWEEP_EXIT_DEF_Q))
            q_ee = float(hist.quantile(SWEEP_ENTER_EXT_Q))
            q_ex = float(hist.quantile(SWEEP_EXIT_EXT_Q))
            if state == "normal":
                if p_now >= q_ee:
                    state = "extreme"
                elif p_now >= q_de:
                    state = "defensive"
            elif state == "defensive":
                if p_now >= q_ee:
                    state = "extreme"
                elif p_now < q_dx:
                    state = "normal"
            elif state == "extreme":
                if p_now < q_ex:
                    state = "defensive"
            exposure = {"normal": SWEEP_EXP_NORMAL, "defensive": SWEEP_EXP_DEF, "extreme": SWEEP_EXP_EXT}[state]
        rows.append({"date": dt, "state": state, "score": p_now, "exposure": exposure})

    if not rows:
        exp_signal = pd.Series(SWEEP_EXP_NORMAL, index=base_r.index, name="exposure_signal")
    else:
        weekly = pd.DataFrame(rows).set_index("date")
        exp_signal = weekly["exposure"].reindex(base_r.index, method="ffill").fillna(SWEEP_EXP_NORMAL).astype(float)
        exp_signal.name = "exposure_signal"

    # Apply signal with a one-trading-day lag.
    exp_applied = exp_signal.shift(1).fillna(SWEEP_EXP_NORMAL).rename("exposure_applied")
    overlay_r = base_r * exp_applied
    trades = int((exp_applied.diff().abs() > 0).sum())
    return exp_signal, exp_applied, overlay_r, trades


# Prefer the full walk-forward PC1 series if it is still in memory; this gives
# the development sweep the earliest possible pre-2018 history without using OOS
# to select candidates.
pc1_source = globals().get("pc1_long", None)
if pc1_source is None:
    pc1_source = pd.concat([dev[BASE_COL], oos[BASE_COL]]).sort_index().drop_duplicates()
else:
    pc1_source = pd.Series(pc1_source).sort_index().drop_duplicates()

base_r_dev = dev[BASE_COL].dropna().astype(float)
base_r_oos = oos[BASE_COL].dropna().astype(float)

state_builders = {
    "RV21":  lambda r: _rolling_rv_sweep(r, window=21, ann=ANN),
    "RV63":  lambda r: _rolling_rv_sweep(r, window=63, ann=ANN),
    "DSV21": lambda r: _rolling_dsv_sweep(r, window=21, ann=ANN),
    "DSV63": lambda r: _rolling_dsv_sweep(r, window=63, ann=ANN),
}

rows = []
best_key_dev = None
best_dev_auc = -np.inf
best_bundle_dev = None

for state_name, builder in state_builders.items():
    state_full = builder(pc1_source).dropna()
    state_dev = state_full.reindex(base_r_dev.index).dropna()
    if state_dev.empty:
        continue

    for H in HORIZON_GRID:
        for tau_q in TAU_GRID:
            tau = float(state_dev.quantile(tau_q))
            score_name = f"p_{state_name}_H{H}_q{int(tau_q*100)}"

            scores_dev = make_ou_scores(base_r_dev.index, state_full, H, tau, score_name)
            exp_sig_dev, exp_app_dev, overlay_dev, trades_dev = build_weekly_overlay_from_scores(scores_dev, base_r_dev)
            perf_dev = _perf(overlay_dev)

            scores_oos = make_ou_scores(base_r_oos.index, state_full, H, tau, score_name)
            exp_sig_oos, exp_app_oos, overlay_oos, trades_oos = build_weekly_overlay_from_scores(scores_oos, base_r_oos)
            perf_oos = _perf(overlay_oos)

            # Selection diagnostics: evaluate the SIGNAL on development, not the
            # overlay P&L. This avoids a degenerate selection when the dev
            # exposure state machine never trades.
            realized_future_dev = state_full.shift(-H).reindex(base_r_dev.index)
            event_dev = (realized_future_dev > tau).astype(float)
            auc_dev = auc_rank(event_dev, scores_dev)
            brier_dev = brier_score(event_dev, scores_dev)

            # OOS diagnostics only. Do not use for model selection.
            realized_future_oos = state_full.shift(-H).reindex(base_r_oos.index)
            event_oos = (realized_future_oos > tau).astype(float)
            auc_oos = auc_rank(event_oos, scores_oos)
            brier_oos = brier_score(event_oos, scores_oos)

            n_score_dev = int(scores_dev.notna().sum())
            row = {
                "state_var": state_name,
                "H": H,
                "tau_q": tau_q,
                "tau": tau,
                "dev_score_obs": n_score_dev,
                "dev_trades": trades_dev,
                "Dev_CAGR": perf_dev["CAGR"],
                "Dev_Sharpe": perf_dev["Sharpe"],
                "Dev_MaxDD": perf_dev["MaxDD"],
                "Dev_AUC": auc_dev,
                "Dev_Brier": brier_dev,
                "dev_event_rate": float(event_dev.mean()),
                "oos_score_mean": float(scores_oos.mean()),
                "oos_event_rate": float(event_oos.mean()),
                "OOS_AUC": auc_oos,
                "OOS_Brier": brier_oos,
                "oos_trades": trades_oos,
                "OOS_CAGR": perf_oos["CAGR"],
                "OOS_Sharpe": perf_oos["Sharpe"],
                "OOS_Sortino": perf_oos["Sortino"],
                "OOS_MaxDD": perf_oos["MaxDD"],
                "OOS_Calmar": perf_oos["Calmar"],
                "OOS_Martin": perf_oos["Martin"],
            }
            rows.append(row)

            valid_for_selection = (
                n_score_dev >= MIN_SELECTION_SCORE_OBS
                and np.isfinite(auc_dev)
            )
            if valid_for_selection and auc_dev > best_dev_auc:
                best_dev_auc = auc_dev
                best_key_dev = (state_name, H, tau_q)
                best_bundle_dev = {
                    "scores_dev": scores_dev,
                    "scores_oos": scores_oos,
                    "event_dev": event_dev,
                    "event_oos": event_oos,
                    "exp_signal_dev": exp_sig_dev,
                    "exp_applied_dev": exp_app_dev,
                    "exp_signal_oos": exp_sig_oos,
                    "exp_applied_oos": exp_app_oos,
                    "overlay_dev": overlay_dev,
                    "overlay_oos": overlay_oos,
                    "state_full": state_full,
                    "tau": tau,
                    "perf_dev": perf_dev,
                    "perf_oos": perf_oos,
                }

ou_sweep = pd.DataFrame(rows)
if not ou_sweep.empty:
    if (ou_sweep["dev_trades"] == 0).all():
        print("WARNING: development overlay P&L is degenerate: dev_trades=0 for every candidate. Selecting by Dev_AUC of the OU exceedance score instead of Dev Sharpe.")
    ou_sweep = ou_sweep.sort_values(
        ["Dev_AUC", "Dev_Brier", "Dev_Sharpe", "OOS_Sharpe"],
        ascending=[False, True, False, False]
    ).reset_index(drop=True)

print("OU slow-layer design sweep — ranked by DEVELOPMENT AUC of the exceedance score")
display(ou_sweep.head(12).round(4))

print("\nOOS diagnostic ranking only — do not use this for model selection")
display(
    ou_sweep.sort_values(["OOS_Sharpe", "OOS_Martin", "OOS_Calmar"], ascending=[False, False, False])
    .head(12).round(4)
)

if best_bundle_dev is not None:
    print(
        f"Selected OU-style challenger by DEV AUC: "
        f"state={best_key_dev[0]}, H={best_key_dev[1]}, tau_q={best_key_dev[2]:.2f}"
    )
    cmp_selected = pd.DataFrame({
        "PC1_base_DEV": _perf(base_r_dev),
        "Selected_OU_DEV": best_bundle_dev["perf_dev"],
        "PC1_base_OOS": _perf(base_r_oos),
        "Selected_OU_OOS": best_bundle_dev["perf_oos"],
    }).T
    display(cmp_selected.round(4))

    ou_sweep_selected = pd.DataFrame({
        "base_r": base_r_oos,
        "score": best_bundle_dev["scores_oos"].reindex(base_r_oos.index),
        "event_proxy": best_bundle_dev["event_oos"].reindex(base_r_oos.index),
        "exposure_signal": best_bundle_dev["exp_signal_oos"].reindex(base_r_oos.index),
        "exposure_applied": best_bundle_dev["exp_applied_oos"].reindex(base_r_oos.index),
        "overlay_r": best_bundle_dev["overlay_oos"].reindex(base_r_oos.index),
    })
    ou_sweep_selected.attrs["selection_rule"] = "max DEV AUC of OU exceedance score, not OOS Sharpe"
    ou_sweep_selected.attrs["selection_caveat"] = globals().get("ou_sweep_selection_caveat", "Dev-AUC selection; disclose development-window limitations.")
    ou_sweep_selected.attrs["selected_state_var"] = best_key_dev[0]
    ou_sweep_selected.attrs["selected_H"] = best_key_dev[1]
    ou_sweep_selected.attrs["selected_tau_q"] = best_key_dev[2]
    ou_sweep_selected.attrs["selected_tau"] = best_bundle_dev["tau"]

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    (1 + base_r_oos).cumprod().plot(ax=axes[0], label="PC1_base")
    (1 + best_bundle_dev["overlay_oos"]).cumprod().plot(ax=axes[0], label="Selected_OU_OOS")
    axes[0].set_title(f"DEV-selected OU challenger — {best_key_dev[0]}, H={best_key_dev[1]}, tau_q={best_key_dev[2]:.2f}")
    axes[0].legend()

    best_bundle_dev["scores_oos"].dropna().plot(ax=axes[1], label="OU-style score")
    axes[1].set_ylabel("probability")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    print("Saved: ou_sweep, ou_sweep_selected")
else:
    ou_sweep_selected = None
    print(
        "No OU sweep candidate had enough development-sample score observations/events "
        f"(min={MIN_SELECTION_SCORE_OBS}) to select by DEV AUC. OOS sweep is diagnostic only; "
        "do not report a best-OOS OU model."
    )
    print("Saved: ou_sweep; ou_sweep_selected=None")


In [ ]:
# ============================================================
# V6c — Purged/embargoed cross-validation for OU specification robustness
# ============================================================
# Purpose:
#   Reduce dependence on a single short/calm development window.
#   The candidate score is evaluated with contiguous time folds and an
#   embargo/purge around each validation block. Because the OU score itself
#   is generated walk-forward and is not supervised on labels, the main role
#   of purging here is to avoid evaluating highly overlapping forward labels
#   as if they were independent.
#
# This is a robustness/model-selection diagnostic. It does not use OOS
# performance to choose the candidate.
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

assert "ou_sweep" in globals(), "Run V6b first."
assert "pc1_source" in globals(), "Run V6b first so that pc1_source exists."
assert "state_builders" in globals(), "Run V6b first so that state_builders exists."
assert "make_ou_scores" in globals(), "Run V6b first so that make_ou_scores exists."
assert "build_weekly_overlay_from_scores" in globals(), "Run V6b first so that overlay builder exists."
assert "base_r_oos" in globals(), "Run V6b first so that base_r_oos exists."
assert "dev" in globals() and "oos" in globals(), "Run the base-panel cell first."

PURGED_CV_SPLITS = 4
PURGED_CV_EMBARGO_DAYS = 10
MIN_PURGED_FOLD_EVENTS = 2


def make_contiguous_time_folds(index: pd.DatetimeIndex, n_splits: int = 4):
    idx = pd.DatetimeIndex(index).sort_values()
    if len(idx) < n_splits:
        raise ValueError("Not enough observations for requested number of folds.")
    parts = np.array_split(np.arange(len(idx)), n_splits)
    return [idx[p] for p in parts if len(p) > 0]


def purged_eval_mask(eval_index: pd.DatetimeIndex, val_index: pd.DatetimeIndex, horizon: int, embargo_days: int = 0):
    """
    Symmetric purge/embargo for overlapping forward labels.

    For a validation block [val_start, val_end], keep only dates whose
    H-day forward label windows stay comfortably inside the block:

        date >= val_start + (H + embargo)
        date <= val_end   - (H + embargo)

    This removes the left-edge and right-edge observations most exposed
    to overlap leakage. If the conservative symmetric cut leaves the fold
    too short, fall back to the raw fold and document the limitation.
    """
    eval_index = pd.DatetimeIndex(eval_index).sort_values()
    val_index = pd.DatetimeIndex(val_index).sort_values()
    if len(val_index) == 0:
        return pd.DatetimeIndex([])

    val_start, val_end = val_index.min(), val_index.max()
    pad = max(int(horizon) + int(embargo_days), 0)
    cutoff_left = val_start + pd.tseries.offsets.BDay(pad)
    cutoff_right = val_end - pd.tseries.offsets.BDay(pad)

    keep = val_index[(val_index >= cutoff_left) & (val_index <= cutoff_right)]

    # If the fold becomes too short after a fully symmetric cut, use the raw
    # fold and explicitly keep the caveat in the notebook narrative.
    if len(keep) < max(5, horizon // 2):
        keep = val_index
    return keep


def ou_purged_cv_candidate(state_name: str, H: int, tau_q: float, cv_index: pd.DatetimeIndex):
    state_full = state_builders[state_name](pc1_source).dropna()
    state_cv = state_full.reindex(cv_index).dropna()
    if len(state_cv) < 30:
        return None
    tau = float(state_cv.quantile(tau_q))
    scores_all = make_ou_scores(cv_index, state_full, H, tau, f"cv_{state_name}_H{H}_q{int(tau_q*100)}")
    future_all = state_full.shift(-H).reindex(cv_index)
    event_all = (future_all > tau).astype(float)

    fold_rows = []
    folds = make_contiguous_time_folds(cv_index, n_splits=PURGED_CV_SPLITS)
    for j, raw_fold_idx in enumerate(folds, start=1):
        val_idx = purged_eval_mask(cv_index, raw_fold_idx, horizon=H, embargo_days=PURGED_CV_EMBARGO_DAYS)
        y = event_all.reindex(val_idx)
        p = scores_all.reindex(val_idx)
        auc = auc_rank(y, p)
        brier = brier_score(y, p)
        fold_rows.append({
            "fold": j,
            "n": int(pd.concat([y, p], axis=1).dropna().shape[0]),
            "event_rate": float(y.mean()) if y.notna().any() else np.nan,
            "events": int(y.sum()) if y.notna().any() else 0,
            "AUC": auc,
            "Brier": brier,
        })
    folds_df = pd.DataFrame(fold_rows)
    valid_auc = folds_df["AUC"].dropna()
    out = {
        "state_var": state_name,
        "H": H,
        "tau_q": tau_q,
        "tau": tau,
        "cv_folds": int(len(folds_df)),
        "cv_obs": int(folds_df["n"].sum()),
        "cv_events": int(folds_df["events"].sum()),
        "CV_AUC_mean": float(valid_auc.mean()) if len(valid_auc) else np.nan,
        "CV_AUC_median": float(valid_auc.median()) if len(valid_auc) else np.nan,
        "CV_Brier_mean": float(folds_df["Brier"].mean()),
        "CV_min_fold_events": int(folds_df["events"].min()) if len(folds_df) else 0,
    }
    return out, folds_df, scores_all, event_all, tau, state_full

# Use the development sample only. This keeps the final OOS period untouched.
cv_index = pd.DatetimeIndex(dev.index).sort_values()
cv_rows = []
cv_details = {}

for state_name in ["RV21", "RV63", "DSV21", "DSV63"]:
    if state_name not in state_builders:
        continue
    for H_cv in HORIZON_GRID:
        for tau_q_cv in TAU_GRID:
            res = ou_purged_cv_candidate(state_name, H_cv, tau_q_cv, cv_index)
            if res is None:
                continue
            row, folds_df, scores_all, event_all, tau, state_full = res
            cv_rows.append(row)
            cv_details[(state_name, H_cv, tau_q_cv)] = {
                "folds": folds_df,
                "scores_all": scores_all,
                "event_all": event_all,
                "tau": tau,
                "state_full": state_full,
            }

ou_purged_cv = pd.DataFrame(cv_rows)
if not ou_purged_cv.empty:
    ou_purged_cv = ou_purged_cv.sort_values(
        ["CV_AUC_mean", "CV_Brier_mean", "CV_AUC_median"],
        ascending=[False, True, False]
    ).reset_index(drop=True)

print("OU purged/embargoed CV ranking — development sample only")
print(f"Folds={PURGED_CV_SPLITS}, embargo={PURGED_CV_EMBARGO_DAYS} business days")
display(ou_purged_cv.head(12).round(4))

# Create a CV-selected OOS overlay object, analogous to ou_sweep_selected.
ou_cv_selected = None
if not ou_purged_cv.empty and np.isfinite(ou_purged_cv.loc[0, "CV_AUC_mean"]):
    best_cv = ou_purged_cv.loc[0]
    key = (best_cv["state_var"], int(best_cv["H"]), float(best_cv["tau_q"]))
    bundle = cv_details[key]
    state_full = bundle["state_full"]
    tau = float(bundle["tau"])
    H_best = int(best_cv["H"])
    score_name = f"cv_selected_{key[0]}_H{H_best}_q{int(float(key[2])*100)}"
    scores_oos = make_ou_scores(base_r_oos.index, state_full, H_best, tau, score_name)
    exp_sig_oos, exp_app_oos, overlay_oos, trades_oos = build_weekly_overlay_from_scores(scores_oos, base_r_oos)
    event_oos = (state_full.shift(-H_best).reindex(base_r_oos.index) > tau).astype(float)

    ou_cv_selected = pd.DataFrame({
        "score": scores_oos,
        "event": event_oos,
        "exposure_signal": exp_sig_oos,
        "exposure_applied": exp_app_oos,
        "base_r": base_r_oos,
        "overlay_r": overlay_oos,
    }).dropna(subset=["base_r"])
    ou_cv_selected.attrs["selection_rule"] = "max purged/embargoed development-sample CV AUC of OU exceedance score"
    ou_cv_selected.attrs["selection_caveat"] = "Purged CV is development-only and data-limited; OOS results remain the final evaluation."
    ou_cv_selected.attrs["selected_state_var"] = key[0]
    ou_cv_selected.attrs["selected_H"] = H_best
    ou_cv_selected.attrs["selected_tau_q"] = float(key[2])
    ou_cv_selected.attrs["selected_tau"] = tau
    ou_cv_selected.attrs["cv_auc_mean"] = float(best_cv["CV_AUC_mean"])
    ou_cv_selected.attrs["cv_brier_mean"] = float(best_cv["CV_Brier_mean"])
    ou_cv_selected.attrs["oos_trades"] = int(trades_oos)

    print(
        "Selected OU by purged CV: "
        f"state={key[0]}, H={H_best}, tau_q={float(key[2]):.2f}, "
        f"CV_AUC={best_cv['CV_AUC_mean']:.4f}, OOS trades={trades_oos}"
    )
    display(pd.DataFrame({
        "PC1_base_OOS": _perf(base_r_oos),
        "OU_CV_selected_OOS": _perf(overlay_oos),
    }).T.round(4))
else:
    print("No valid purged-CV OU candidate selected; ou_cv_selected=None")

print("\nSaved: ou_purged_cv, ou_cv_selected")


In [ ]:
# ============================================================
# V6d — Rough-volatility diagnostic: rolling Hurst estimate on log RV21
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

assert "pc1_source" in globals(), "Run V6b first so that pc1_source exists."
assert "state_builders" in globals() and "RV21" in state_builders, "Run V6b first so that state_builders['RV21'] exists."
assert "oos" in globals() and "dev" in globals(), "Run the base-panel cell first."

ROUGH_EST_WINDOW = 504   # ~2y
ROUGH_LAGS = [1, 2, 5, 10, 21]
ROUGH_MIN_OBS = 252

def hurst_variogram_estimator(x: pd.Series, lags: list[int] = ROUGH_LAGS):
    """
    Simple variogram-style H estimator:
        Var(x_{t+h} - x_t) ~ h^(2H)
    Regress log empirical variance of increments on log lag and set H = slope/2.
    Returns H and the regression R^2 as a rough fit diagnostic.
    """
    x = pd.Series(x).dropna().astype(float)
    vals = []
    for h in lags:
        dx = x.diff(h).dropna()
        if len(dx) < max(20, 3 * h):
            continue
        v = float(np.var(dx, ddof=1))
        if np.isfinite(v) and v > 0:
            vals.append((h, v))
    if len(vals) < 3:
        return np.nan, np.nan
    log_h = np.log([v[0] for v in vals])
    log_v = np.log([v[1] for v in vals])
    slope, intercept = np.polyfit(log_h, log_v, 1)
    fit = intercept + slope * log_h
    ss_res = float(np.sum((log_v - fit) ** 2))
    ss_tot = float(np.sum((log_v - log_v.mean()) ** 2))
    r2 = np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot
    return float(0.5 * slope), float(r2)

rv21_full = state_builders["RV21"](pc1_source).dropna()
log_rv21_full = np.log(rv21_full[rv21_full > 0]).rename("log_rv21")

rows = []
for dt in log_rv21_full.index:
    hist = log_rv21_full.loc[:dt].dropna().tail(ROUGH_EST_WINDOW)
    if len(hist) < ROUGH_MIN_OBS:
        continue
    H_hat, r2 = hurst_variogram_estimator(hist, lags=ROUGH_LAGS)
    rows.append({"date": dt, "H_hat": H_hat, "fit_r2": r2, "window_obs": len(hist)})

rough_hurst = pd.DataFrame(rows).set_index("date") if rows else pd.DataFrame(columns=["H_hat", "fit_r2", "window_obs"])

print("Walk-forward rough-vol diagnostic on log RV21")
print(f"Window={ROUGH_EST_WINDOW} obs | lags={ROUGH_LAGS}")
if rough_hurst.empty:
    print("No H estimates available — check sample length and state construction.")
else:
    display(rough_hurst.describe().round(4))

    dev_idx = rough_hurst.index.intersection(dev.index)
    oos_idx = rough_hurst.index.intersection(oos.index)

    def _summ(s: pd.Series):
        s = pd.Series(s).dropna()
        if s.empty:
            return {"n": 0, "mean_H": np.nan, "median_H": np.nan, "pct_below_05": np.nan, "pct_below_04": np.nan}
        return {
            "n": int(len(s)),
            "mean_H": float(s.mean()),
            "median_H": float(s.median()),
            "pct_below_05": float((s < 0.5).mean()),
            "pct_below_04": float((s < 0.4).mean()),
        }

    rough_summary = pd.DataFrame({
        "DEV": _summ(rough_hurst.loc[dev_idx, "H_hat"]),
        "OOS": _summ(rough_hurst.loc[oos_idx, "H_hat"]),
        "FULL": _summ(rough_hurst["H_hat"]),
    }).T
    print("\nRough-vol summary:")
    display(rough_summary.round(4))

    recommendation = "OU retained; rough-vol challenger not yet empirically compelled."
    if rough_summary.loc["FULL", "pct_below_05"] >= 0.70 and rough_summary.loc["FULL", "median_H"] < 0.5:
        recommendation = "Rough-vol challenger is empirically motivated (H < 0.5 stably)."
    elif rough_summary.loc["FULL", "pct_below_05"] >= 0.50:
        recommendation = "Rough-vol challenger is plausible; test against OU/HAR in a dedicated slow-layer sweep."

    rough_hurst.attrs["recommendation"] = recommendation
    rough_hurst.attrs["window_obs"] = ROUGH_EST_WINDOW
    rough_hurst.attrs["lags"] = tuple(ROUGH_LAGS)

    print(f"\nRecommendation: {recommendation}")

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    rough_hurst["H_hat"].plot(ax=axes[0], title="Walk-forward Hurst estimate on log RV21")
    axes[0].axhline(0.5, color="red", ls="--", lw=1.0)
    axes[0].axhline(0.4, color="orange", ls="--", lw=1.0)
    axes[0].set_ylabel("H")
    rough_hurst["fit_r2"].plot(ax=axes[1], title="Variogram fit R²")
    axes[1].set_ylabel("R²")
    plt.tight_layout()
    plt.show()

print("\nSaved: rough_hurst")


In [ ]:
# ============================================================
# V6e — CIR slow-layer challenger for the OU retained overlay
# ============================================================
# Purpose:
#   Test a positive-variance stochastic-volatility-style challenger against the
#   retained OU/log-volatility slow layer. The CIR layer is NOT selected on OOS
#   performance. A small design grid is ranked by development-sample AUC of the
#   forward variance-exceedance score.
#
# Model:
#   dV_t = kappa(theta - V_t)dt + sigma sqrt(V_t)dW_t
#
# Implementation:
#   - V_t is the square of the realized-volatility state (RV21 by default);
#   - parameters are estimated on expanding history with an Euler regression;
#   - H-step exceedance probabilities use the exact noncentral chi-square
#     transition when parameters are admissible;
#   - the overlay uses the same weekly hysteretic mapping as the OU sweep.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import ncx2
from IPython.display import display
import matplotlib.pyplot as plt

assert "pc1_source" in globals(), "Run V6b first so that pc1_source exists."
assert "state_builders" in globals() and "RV21" in state_builders, "Run V6b first."
assert "build_weekly_overlay_from_scores" in globals(), "Run V6b first."
assert "auc_rank" in globals(), "Run V6b first so that auc_rank exists."
assert "brier_score" in globals(), "Run V6b first so that brier_score exists."
assert "dev" in globals() and "oos" in globals(), "Run the base-panel cell first."

CIR_H_GRID = [21, 42, 63]
CIR_TAU_Q_GRID = [0.80, 0.85]
CIR_MIN_CALIB_OBS = 252
CIR_STATE_NAME = "RV21"
CIR_EPS = 1e-10


def fit_cir_euler(var_series: pd.Series):
    """Estimate CIR parameters from an expanding positive variance series.

    Euler regression:
        ΔV_t = a + b V_t + eps_t,  kappa=-b, theta=a/kappa.
    sigma is estimated from eps_t / sqrt(V_t).
    """
    v = pd.Series(var_series).dropna().astype(float).clip(lower=CIR_EPS)
    if len(v) < 60:
        return None
    v0 = v.iloc[:-1].values
    dv = (v.iloc[1:].values - v.iloc[:-1].values)
    X = np.column_stack([np.ones_like(v0), v0])
    try:
        beta = np.linalg.lstsq(X, dv, rcond=None)[0]
    except Exception:
        return None
    a, b = float(beta[0]), float(beta[1])
    kappa = -b
    if not np.isfinite(kappa) or kappa <= 0:
        return None
    theta = a / kappa
    if not np.isfinite(theta) or theta <= 0:
        return None
    resid = dv - X @ beta
    sig2 = float(np.mean((resid ** 2) / np.maximum(v0, CIR_EPS)))
    sigma = float(np.sqrt(max(sig2, CIR_EPS)))
    if not np.isfinite(sigma) or sigma <= 0:
        return None
    return {"kappa": kappa, "theta": theta, "sigma": sigma}


def cir_exceedance_prob(v_now: float, tau_v: float, params: dict, H: int):
    """Exact CIR transition exceedance probability P(V_{t+H} > tau_v | F_t)."""
    v_now = float(max(v_now, CIR_EPS))
    tau_v = float(max(tau_v, CIR_EPS))
    kappa, theta, sigma = params["kappa"], params["theta"], params["sigma"]
    exp_kH = np.exp(-kappa * H)
    denom = sigma ** 2 * (1.0 - exp_kH)
    if denom <= 0:
        return np.nan
    c = denom / (4.0 * kappa)
    df = 4.0 * kappa * theta / (sigma ** 2)
    lam = 4.0 * kappa * exp_kH * v_now / denom
    if not np.isfinite(c) or not np.isfinite(df) or not np.isfinite(lam) or c <= 0 or df <= 0:
        return np.nan
    return float(ncx2.sf(tau_v / c, df, lam))


def make_cir_scores(eval_index: pd.DatetimeIndex, rv_state_full: pd.Series, H: int, tau_rv: float, name: str):
    var_full = (pd.Series(rv_state_full).dropna().astype(float) ** 2).clip(lower=CIR_EPS)
    tau_v = float(max(tau_rv ** 2, CIR_EPS))
    scores = pd.Series(np.nan, index=pd.DatetimeIndex(eval_index), name=name)
    for dt in pd.DatetimeIndex(eval_index):
        hist = var_full.loc[:dt].dropna()
        if len(hist) < CIR_MIN_CALIB_OBS:
            continue
        params = fit_cir_euler(hist)
        if params is None:
            continue
        scores.loc[dt] = cir_exceedance_prob(float(hist.iloc[-1]), tau_v, params, H)
    return scores


rv_state_full = state_builders[CIR_STATE_NAME](pc1_source).dropna()
base_r_dev = dev[BASE_COL].dropna().astype(float)
base_r_oos = oos[BASE_COL].dropna().astype(float)
full_eval_index = pd.DatetimeIndex(pd.concat([base_r_dev, base_r_oos]).sort_index().index)

cir_rows = []
cir_bundles = {}
for H in CIR_H_GRID:
    for tau_q in CIR_TAU_Q_GRID:
        rv_dev_state = rv_state_full.reindex(dev.index).dropna()
        tau_rv = float(rv_dev_state.quantile(tau_q))
        scores_full = make_cir_scores(full_eval_index, rv_state_full, H, tau_rv, f"CIR_{CIR_STATE_NAME}_H{H}_q{int(100*tau_q)}")
        future_dev = rv_state_full.shift(-H).reindex(dev.index)
        event_dev = (future_dev > tau_rv).astype(float)
        dev_scores = scores_full.reindex(dev.index)
        auc_dev = auc_rank(event_dev, dev_scores)
        brier_dev = brier_score(event_dev, dev_scores)

        sig_dev, exp_dev, r_dev, trades_dev = build_weekly_overlay_from_scores(dev_scores, base_r_dev)
        sig_oos, exp_oos, r_oos, trades_oos = build_weekly_overlay_from_scores(scores_full.reindex(oos.index), base_r_oos)
        ps_dev = perf_stats(r_dev) if "perf_stats" in globals() else _perf(r_dev)
        ps_oos = perf_stats(r_oos) if "perf_stats" in globals() else _perf(r_oos)

        key = (H, tau_q)
        cir_rows.append({
            "H": H,
            "tau_q": tau_q,
            "tau_rv": tau_rv,
            "Dev_AUC": auc_dev,
            "Dev_Brier": brier_dev,
            "dev_trades": trades_dev,
            "Dev_Sharpe": ps_dev.get("Sharpe", np.nan),
            "Dev_Martin": ps_dev.get("Martin", np.nan),
            "OOS_CAGR": ps_oos.get("CAGR", np.nan),
            "OOS_Sharpe": ps_oos.get("Sharpe", np.nan),
            "OOS_Martin": ps_oos.get("Martin", np.nan),
            "OOS_MaxDD": ps_oos.get("MaxDD", np.nan),
            "oos_trades": trades_oos,
        })
        cir_bundles[key] = {
            "scores_full": scores_full,
            "tau_rv": tau_rv,
            "dev": (sig_dev, exp_dev, r_dev),
            "oos": (sig_oos, exp_oos, r_oos),
        }

cir_sweep = pd.DataFrame(cir_rows).sort_values(["Dev_AUC", "Dev_Brier"], ascending=[False, True]).reset_index(drop=True)
print("CIR slow-layer challenger sweep — ranked by DEVELOPMENT AUC")
display(cir_sweep.round(4))

best_cir = cir_sweep.iloc[0]
best_key = (int(best_cir["H"]), float(best_cir["tau_q"]))
best_bundle = cir_bundles[best_key]

sig_full, exp_full, r_full, trades_full = build_weekly_overlay_from_scores(best_bundle["scores_full"], pd.concat([base_r_dev, base_r_oos]).sort_index())
cir_challenger_panel_full = pd.DataFrame({
    "exposure_signal": sig_full,
    "exposure_applied": exp_full,
    "overlay_r_gross": r_full,
})
cir_challenger_panel_full.attrs.update({
    "selection_rule": "max development-sample AUC of CIR exceedance score",
    "selected_state_var": CIR_STATE_NAME,
    "selected_H": int(best_cir["H"]),
    "selected_tau_q": float(best_cir["tau_q"]),
    "selected_tau_rv": float(best_cir["tau_rv"]),
})

print("\nSelected CIR challenger:")
display(pd.DataFrame([cir_challenger_panel_full.attrs]).T.rename(columns={0: "value"}))
print("\nSaved: cir_sweep, cir_challenger_panel_full")


## 6) Fast-layer challengers: Kalman/Mahalanobis robust, hybrid, and gated variants

In [ ]:
# ============================================================
# V7b — Kalman/Mahalanobis fast overlay
# Robust variant  : local-level Kalman + backward/current NIS
# Hybrid variant  : local-level past detector + VAR(1) forward detector
#                   combined through weighted statistical severities
#
# Final policy:
#   - theoretical chi-square upper-tail p-values
#   - continuous tail scaling on the relevant tail only
#   - first tail segment damped by alpha in (0, 1]
#   - actual exposure adjusts with severity-dependent speed
#   - one-day implementation lag
#   - Corwin--Schultz incremental costs, no flat-bps fallback
#
# The robust variant calibrates alpha on the DEVELOPMENT sample by
# maximizing Martin_net.
#
# The hybrid variant calibrates (omega, alpha) on the DEVELOPMENT sample
# by maximizing Martin_net, where:
#   - omega weights the backward/current detector
#   - (1-omega) weights the forward-looking VAR(1) detector
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2
from IPython.display import display
import matplotlib.pyplot as plt

assert "iv" in globals(), "Run the data section first so that iv exists."
assert "dev" in globals() and "oos" in globals(), "Run the base-panel cell first so that dev/oos exist."
assert "W_hist_long" in globals(), "Run the base-panel cell first so that W_hist_long exists."
assert "asset_costs" in globals(), "Run the Corwin--Schultz cost block first so that asset_costs exists."
assert "px_stocks" in globals(), "Run the data section first so that px_stocks exists."

# -------------------------
# Configuration
# -------------------------
KALMAN_SIGNAL_COLS = ["log_iv", "dlog_iv_plus", "dlog_iv_minus"]

# Past/current detector
KALMAN_LAMBDA_LEVEL = 0.02
KALMAN_COV_SHRINK = 0.10
KALMAN_INIT_OBS = 126
KALMAN_ROLL_WINDOW = 5

# Forward detector
KALMAN_H_FWD = 2
KALMAN_VAR_WINDOW = 252
KALMAN_VAR_MIN_OBS = 126
KALMAN_VAR_RIDGE = 1e-6

# Tail thresholds (upper-tail p-values of the theoretical chi-square law)
KALMAN_P_ENTER = 0.0100   # start de-risking
KALMAN_P_MID   = 0.0025   # stronger part of the tail
KALMAN_P_FLOOR = 0.0005   # saturation

# Exposure levels
KALMAN_EXP_NORMAL = 1.00
KALMAN_EXP_MID    = 0.85
KALMAN_EXP_MIN    = 0.50

# Exposure-adjustment speed by severity segment
KALMAN_ETA_MILD    = 0.25
KALMAN_ETA_MID     = 0.60
KALMAN_ETA_EXTREME = 1.00

# Friction controls
KALMAN_MIN_EXPOSURE_MOVE = 0.03
KALMAN_MIN_HOLD_DAYS = 3

# Calibration grids
KALMAN_ALPHA_GRID = [0.25, 0.50, 0.75, 1.00]
KALMAN_OMEGA_GRID = [0.25, 0.50, 0.75]

ANN = 252
PC_CORE = globals().get("PC_CORE", "PC1")
BASE_COL = globals().get("BASE_COL", "pc1_base")

# -------------------------
# Helpers
# -------------------------
def build_kalman_iv_signals(iv_level: pd.Series, target_index: pd.DatetimeIndex) -> pd.DataFrame:
    iv_s = pd.Series(iv_level).astype(float).reindex(target_index).ffill()
    log_iv = np.log(iv_s)
    dlog = log_iv.diff()
    z = pd.DataFrame({
        "log_iv": log_iv,
        "dlog_iv_plus": dlog.clip(lower=0.0),
        "dlog_iv_minus": dlog.clip(upper=0.0),
    }, index=target_index)
    return z.dropna()

def regularized_cov(x: np.ndarray, shrink: float = 0.10, eps: float = 1e-10) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    cov = np.cov(x, rowvar=False, ddof=1)
    if cov.ndim == 0:
        cov = np.array([[float(cov)]])
    diag = np.diag(np.diag(cov))
    out = (1.0 - shrink) * cov + shrink * diag
    out = out + eps * np.eye(out.shape[0])
    return out

def inv_psd(A: np.ndarray) -> np.ndarray:
    return np.linalg.pinv(np.asarray(A, dtype=float), hermitian=True)

def perf_stats(r: pd.Series, ann: int = ANN) -> dict:
    r = pd.Series(r).dropna().astype(float)
    if len(r) < 5:
        return {"n": len(r), "CAGR": np.nan, "AnnVol": np.nan, "Sharpe": np.nan,
                "Sortino": np.nan, "MaxDD": np.nan, "Calmar": np.nan, "Martin": np.nan}
    w = (1.0 + r).cumprod()
    n = len(r); yrs = n / ann
    cagr = w.iloc[-1] ** (1.0 / yrs) - 1.0 if yrs > 0 else np.nan
    vol = r.std(ddof=1) * np.sqrt(ann)
    mu = r.mean() * ann
    shr = mu / vol if vol > 0 else np.nan
    dn = r[r < 0]
    dvol = dn.std(ddof=1) * np.sqrt(ann) if len(dn) > 1 else np.nan
    srt = mu / dvol if pd.notna(dvol) and dvol > 0 else np.nan
    dd = w / w.cummax() - 1.0
    mdd = dd.min()
    cal = cagr / abs(mdd) if mdd < 0 else np.nan
    ulc = np.sqrt(np.mean((100.0 * dd) ** 2)) / 100.0
    mar = cagr / ulc if ulc > 0 else np.nan
    return {"n": n, "CAGR": cagr, "AnnVol": vol, "Sharpe": shr,
            "Sortino": srt, "MaxDD": mdd, "Calmar": cal, "Martin": mar}

def factor_weights_daily_from_history(weights_hist, factor_name: str, target_index: pd.DatetimeIndex, tickers: list[str]) -> pd.DataFrame:
    rows = []
    for item in weights_hist:
        if not isinstance(item, dict):
            continue
        reb = pd.Timestamp(item["rebalance"])
        W = item["W"].copy()
        if factor_name not in W.columns:
            continue
        s = W[factor_name].reindex(tickers).astype(float)
        s.name = reb
        rows.append(s)
    if not rows:
        raise ValueError(f"No weights found for factor {factor_name} in W_hist_long.")
    W_step = pd.DataFrame(rows)
    W_step.index = pd.to_datetime(W_step.index)
    W_step = W_step.sort_index()
    return W_step.reindex(target_index, method="ffill")

def basket_oneway_cost_series(target_index: pd.DatetimeIndex, pc_core: str = PC_CORE) -> pd.Series:
    tickers = list(px_stocks.columns)
    W_pc_daily = factor_weights_daily_from_history(W_hist_long, pc_core, target_index, tickers)
    cost_panel = asset_costs.reindex(index=target_index, columns=tickers).ffill().fillna(0.0)
    gross_abs = W_pc_daily.abs().sum(axis=1).replace(0.0, np.nan)
    portfolio_oneway_cost = ((W_pc_daily.abs() * cost_panel).sum(axis=1) / gross_abs).fillna(0.0)
    portfolio_oneway_cost.name = "portfolio_oneway_cost_CS"
    return portfolio_oneway_cost

# -------------------------
# Backward/current score construction
# -------------------------
def kalman_local_level_scores(
    z_panel: pd.DataFrame,
    start_eval: pd.Timestamp,
    init_obs: int = KALMAN_INIT_OBS,
    lambda_level: float = KALMAN_LAMBDA_LEVEL,
    cov_shrink: float = KALMAN_COV_SHRINK,
    roll_window: int = KALMAN_ROLL_WINDOW,
) -> pd.DataFrame:
    z_panel = z_panel.dropna().astype(float).sort_index()
    if len(z_panel) < init_obs + roll_window:
        raise ValueError("Kalman signal panel too short.")

    K = z_panel.shape[1]
    z_init = z_panel.iloc[:init_obs]
    R = regularized_cov(z_init.values, shrink=cov_shrink)
    Q = lambda_level * R
    x = z_init.mean(axis=0).values.astype(float)
    P = R.copy()

    d2_hist = []
    rows = []
    for dt, row in z_panel.iloc[init_obs:].iterrows():
        z_t = row.values.astype(float)

        x_pred = x.copy()
        P_pred = P + Q

        nu = z_t - x_pred
        S = P_pred + R
        d2 = float(nu @ inv_psd(S) @ nu)
        d2_hist.append(d2)

        K_gain = P_pred @ inv_psd(S)
        x = x_pred + K_gain @ nu
        P = (np.eye(K) - K_gain) @ P_pred

        if len(d2_hist) >= roll_window:
            score_sum = float(np.sum(d2_hist[-roll_window:]))
            p_value = float(chi2.sf(score_sum, df=K * roll_window))
        else:
            score_sum = np.nan
            p_value = np.nan

        rows.append({"date": dt, "score_sum_past": score_sum, "p_past": p_value, "d2_bwd": d2})

    out = pd.DataFrame(rows).set_index("date")
    return out.loc[out.index >= pd.Timestamp(start_eval)].copy()

# -------------------------
# Forward score construction (VAR(1) + Mahalanobis vs normal center)
# -------------------------
def fit_var1_ols(hist: pd.DataFrame, cov_shrink: float = KALMAN_COV_SHRINK, ridge: float = KALMAN_VAR_RIDGE):
    hist = pd.DataFrame(hist).dropna().astype(float)
    K = hist.shape[1]
    if len(hist) < 3:
        return None

    Xlag = hist.iloc[:-1].values
    Y = hist.iloc[1:].values
    Xreg = np.column_stack([np.ones(len(Xlag)), Xlag])
    XtX = Xreg.T @ Xreg + ridge * np.eye(Xreg.shape[1])
    Beta = np.linalg.solve(XtX, Xreg.T @ Y)   # (K+1) x K
    c = Beta[0]                               # K
    A = Beta[1:].T                            # K x K

    resid = Y - Xreg @ Beta
    if len(resid) < 2:
        Omega = regularized_cov(np.vstack([np.zeros(K), np.zeros(K)]), shrink=cov_shrink)
    else:
        Omega = regularized_cov(resid, shrink=cov_shrink)
    return c, A, Omega

def var1_forward_scores(
    z_panel: pd.DataFrame,
    start_eval: pd.Timestamp,
    init_obs: int = KALMAN_INIT_OBS,
    h_fwd: int = KALMAN_H_FWD,
    var_window: int = KALMAN_VAR_WINDOW,
    var_min_obs: int = KALMAN_VAR_MIN_OBS,
    cov_shrink: float = KALMAN_COV_SHRINK,
) -> pd.DataFrame:
    z_panel = z_panel.dropna().astype(float).sort_index()
    K = z_panel.shape[1]
    if len(z_panel) < max(init_obs, var_min_obs) + h_fwd + 1:
        raise ValueError("Kalman/VAR signal panel too short for forward variant.")

    rows = []
    for loc in range(init_obs, len(z_panel)):
        dt = z_panel.index[loc]
        hist = z_panel.iloc[max(0, loc - var_window + 1):loc + 1].copy()
        if len(hist) < var_min_obs:
            rows.append({"date": dt, "score_sum_fwd": np.nan, "p_fwd": np.nan, "d2_fwd_mean": np.nan})
            continue

        fit = fit_var1_ols(hist, cov_shrink=cov_shrink)
        if fit is None:
            rows.append({"date": dt, "score_sum_fwd": np.nan, "p_fwd": np.nan, "d2_fwd_mean": np.nan})
            continue

        c, A, Omega = fit
        z_curr = hist.iloc[-1].values.astype(float)
        normal_center = hist.mean(axis=0).values.astype(float)

        mu = z_curr.copy()
        Sigma = np.zeros((K, K), dtype=float)
        d2_fwd = []
        for _ in range(1, h_fwd + 1):
            mu = c + A @ mu
            Sigma = A @ Sigma @ A.T + Omega
            diff = mu - normal_center
            d2 = float(diff @ inv_psd(Sigma + 1e-10 * np.eye(K)) @ diff)
            d2_fwd.append(d2)

        score_sum = float(np.sum(d2_fwd))
        p_fwd = float(chi2.sf(score_sum, df=K * h_fwd))
        rows.append({
            "date": dt,
            "score_sum_fwd": score_sum,
            "p_fwd": p_fwd,
            "d2_fwd_mean": float(np.mean(d2_fwd)),
        })

    out = pd.DataFrame(rows).set_index("date")
    return out.loc[out.index >= pd.Timestamp(start_eval)].copy()

# -------------------------
# p-value/severity combination and policy
# -------------------------
def combine_pvalues_geomean(p_past: float, p_fwd: float, omega: float) -> float:
    if pd.isna(p_past) and pd.isna(p_fwd):
        return np.nan
    if pd.isna(p_past):
        return float(p_fwd)
    if pd.isna(p_fwd):
        return float(p_past)
    p_past = float(max(min(p_past, 1.0), 1e-12))
    p_fwd = float(max(min(p_fwd, 1.0), 1e-12))
    s = float(omega) * (-np.log(p_past)) + (1.0 - float(omega)) * (-np.log(p_fwd))
    return float(np.exp(-s))

def fast_target_from_pvalue(p: float, alpha: float) -> float:
    if pd.isna(p):
        return KALMAN_EXP_NORMAL
    p = float(max(min(p, 1.0), 1e-12))

    if p > KALMAN_P_ENTER:
        return KALMAN_EXP_NORMAL

    w_mid_actual = 1.0 - float(alpha) * (1.0 - KALMAN_EXP_MID)

    if p > KALMAN_P_MID:
        u1 = (np.log(KALMAN_P_ENTER) - np.log(p)) / (np.log(KALMAN_P_ENTER) - np.log(KALMAN_P_MID))
        return float(1.0 - (1.0 - w_mid_actual) * u1)

    if p > KALMAN_P_FLOOR:
        u2 = (np.log(KALMAN_P_MID) - np.log(p)) / (np.log(KALMAN_P_MID) - np.log(KALMAN_P_FLOOR))
        return float(w_mid_actual - (w_mid_actual - KALMAN_EXP_MIN) * u2)

    return float(KALMAN_EXP_MIN)

def severity_eta(p: float) -> float:
    if pd.isna(p):
        return 0.0
    if p <= KALMAN_P_FLOOR:
        return KALMAN_ETA_EXTREME
    if p <= KALMAN_P_MID:
        return KALMAN_ETA_MID
    if p <= KALMAN_P_ENTER:
        return KALMAN_ETA_MILD
    return KALMAN_ETA_MILD

def actual_exposure_from_targets(target_exp: pd.Series, p_values: pd.Series) -> pd.Series:
    target_exp = pd.Series(target_exp).astype(float)
    p_values = pd.Series(p_values).reindex(target_exp.index)
    out = pd.Series(index=target_exp.index, dtype=float)
    prev = KALMAN_EXP_NORMAL
    days_since_change = 999

    for dt in target_exp.index:
        tgt = float(target_exp.loc[dt]) if pd.notna(target_exp.loc[dt]) else prev
        p = float(p_values.loc[dt]) if pd.notna(p_values.loc[dt]) else np.nan
        eta = severity_eta(p)
        candidate = prev + eta * (tgt - prev)

        if abs(candidate - prev) < KALMAN_MIN_EXPOSURE_MOVE and not (pd.notna(p) and p <= KALMAN_P_FLOOR):
            candidate = prev

        if days_since_change < KALMAN_MIN_HOLD_DAYS and abs(candidate - prev) > 0 and not (pd.notna(p) and p <= KALMAN_P_FLOOR):
            candidate = prev

        candidate = float(np.clip(candidate, KALMAN_EXP_MIN, KALMAN_EXP_NORMAL))
        out.loc[dt] = candidate

        if abs(candidate - prev) > 1e-12:
            days_since_change = 0
        else:
            days_since_change += 1
        prev = candidate

    return out.rename("exposure_signal")

# -------------------------
# Backtest helpers
# -------------------------
def compute_overlay_returns(base_r: pd.Series, exposure_signal: pd.Series, basket_cost: pd.Series):
    idx = pd.DatetimeIndex(base_r.index).intersection(exposure_signal.index).intersection(basket_cost.index)
    base = pd.Series(base_r).reindex(idx).astype(float)
    exp_signal = pd.Series(exposure_signal).reindex(idx).fillna(KALMAN_EXP_NORMAL).astype(float)
    exp_applied = exp_signal.shift(1).fillna(KALMAN_EXP_NORMAL).rename("exposure_applied")
    overlay_turnover = exp_applied.diff().abs().fillna(0.0).rename("overlay_turnover")
    tc = (overlay_turnover * basket_cost.reindex(idx).fillna(0.0)).rename("overlay_cost_CS")
    gross = (exp_applied * base).rename("overlay_r_gross")
    net = (gross - tc).rename("overlay_r_net_CS")
    panel = pd.concat([base.rename("base_r"), exp_signal.rename("exposure_signal"), exp_applied, overlay_turnover,
                       basket_cost.reindex(idx), tc, gross, net], axis=1)
    return panel

def calibrate_alpha_dev(scores: pd.DataFrame, base_r_dev: pd.Series, basket_cost_dev: pd.Series, alpha_grid=None):
    if alpha_grid is None:
        alpha_grid = KALMAN_ALPHA_GRID
    rows = []
    panels = {}
    common = pd.DatetimeIndex(base_r_dev.index).intersection(scores.index)
    scores_sub = scores.loc[common].dropna(subset=["p_value"])
    for alpha in alpha_grid:
        target = scores_sub["p_value"].apply(lambda p: fast_target_from_pvalue(p, alpha)).rename("target_exposure")
        exp_signal = actual_exposure_from_targets(target, scores_sub["p_value"])
        panel = compute_overlay_returns(base_r_dev.reindex(scores_sub.index), exp_signal, basket_cost_dev.reindex(scores_sub.index))
        ps = perf_stats(panel["overlay_r_net_CS"])
        rows.append({
            "alpha": float(alpha),
            "n_trades": int((panel["overlay_turnover"] > 0).sum()),
            "total_tc_pct": float(100.0 * panel["overlay_cost_CS"].sum()),
            **ps,
        })
        panels[float(alpha)] = panel
    calib = pd.DataFrame(rows).sort_values(["Martin", "Sharpe", "total_tc_pct"], ascending=[False, False, True]).reset_index(drop=True)
    best_alpha = float(calib.loc[0, "alpha"])
    return calib, best_alpha, panels[best_alpha]

def calibrate_hybrid_dev(scores: pd.DataFrame, base_r_dev: pd.Series, basket_cost_dev: pd.Series,
                         alpha_grid=None, omega_grid=None):
    if alpha_grid is None:
        alpha_grid = KALMAN_ALPHA_GRID
    if omega_grid is None:
        omega_grid = KALMAN_OMEGA_GRID
    rows = []
    panels = {}
    common = pd.DatetimeIndex(base_r_dev.index).intersection(scores.index)
    scores_sub = scores.loc[common].dropna(subset=["p_past", "p_fwd"])
    for omega in omega_grid:
        p_comb = pd.Series(
            [combine_pvalues_geomean(pp, pf, omega) for pp, pf in zip(scores_sub["p_past"], scores_sub["p_fwd"])],
            index=scores_sub.index,
            name="p_value"
        )
        for alpha in alpha_grid:
            target = p_comb.apply(lambda p: fast_target_from_pvalue(p, alpha)).rename("target_exposure")
            exp_signal = actual_exposure_from_targets(target, p_comb)
            panel = compute_overlay_returns(base_r_dev.reindex(scores_sub.index), exp_signal, basket_cost_dev.reindex(scores_sub.index))
            ps = perf_stats(panel["overlay_r_net_CS"])
            rows.append({
                "omega": float(omega),
                "alpha": float(alpha),
                "n_trades": int((panel["overlay_turnover"] > 0).sum()),
                "total_tc_pct": float(100.0 * panel["overlay_cost_CS"].sum()),
                **ps,
            })
            panels[(float(omega), float(alpha))] = panel

    calib = pd.DataFrame(rows).sort_values(["Martin", "Sharpe", "total_tc_pct"], ascending=[False, False, True]).reset_index(drop=True)
    best_omega = float(calib.loc[0, "omega"])
    best_alpha = float(calib.loc[0, "alpha"])
    return calib, best_omega, best_alpha, panels[(best_omega, best_alpha)]

def run_kalman_robust(scores_past: pd.DataFrame, base_r_full: pd.Series, basket_cost_full: pd.Series):
    base_r_full = pd.Series(base_r_full).dropna().astype(float).sort_index()
    basket_cost_full = pd.Series(basket_cost_full).reindex(base_r_full.index).ffill().fillna(0.0)

    dev_idx = pd.DatetimeIndex(dev.index).intersection(scores_past.index).intersection(base_r_full.index)
    oos_idx = pd.DatetimeIndex(oos.index).intersection(scores_past.index).intersection(base_r_full.index)

    calib_table, best_alpha, best_dev_panel = calibrate_alpha_dev(
        scores=scores_past.rename(columns={"p_past": "p_value"}),
        base_r_dev=base_r_full.reindex(dev_idx),
        basket_cost_dev=basket_cost_full.reindex(dev_idx),
        alpha_grid=KALMAN_ALPHA_GRID,
    )

    p_full = scores_past["p_past"].copy()
    target_full = p_full.apply(lambda p: fast_target_from_pvalue(p, best_alpha)).rename("target_exposure")
    exp_signal_full = actual_exposure_from_targets(target_full, p_full)
    valid_idx = p_full.index.intersection(base_r_full.index)
    panel_full = compute_overlay_returns(base_r_full.reindex(valid_idx), exp_signal_full.reindex(valid_idx), basket_cost_full.reindex(valid_idx))
    panel_oos = panel_full.loc[panel_full.index.intersection(oos_idx)].copy()

    perf = pd.DataFrame({
        "PC1_base": perf_stats(panel_oos["base_r"]),
        "Fast_Kalman_robust_gross": perf_stats(panel_oos["overlay_r_gross"]),
        "Fast_Kalman_robust_net_CS": perf_stats(panel_oos["overlay_r_net_CS"]),
    }).T

    out = {
        "name": "Fast_Kalman_robust",
        "scores": scores_past,
        "alpha_grid": KALMAN_ALPHA_GRID,
        "calibration_table": calib_table,
        "selected_alpha": best_alpha,
        "dev_panel": best_dev_panel,
        "dev_objective": float(calib_table.loc[0, "Martin"]) if len(calib_table) else np.nan,
        "dev_total_tc_pct": float(calib_table.loc[0, "total_tc_pct"]) if len(calib_table) else np.nan,
        "dev_n_trades": int(calib_table.loc[0, "n_trades"]) if len(calib_table) else np.nan,
        "panel_full": panel_full,
        "panel_oos": panel_oos,
        "base_r": panel_oos["base_r"],
        "overlay_r_gross": panel_oos["overlay_r_gross"],
        "overlay_r_net": panel_oos["overlay_r_net_CS"],
        "signal_exposure_daily": panel_oos["exposure_signal"],
        "exposure_daily": panel_oos["exposure_applied"],
        "tc_daily": panel_oos["overlay_cost_CS"],
        "perf_table": perf,
        "oos_trades": int((panel_oos["overlay_turnover"] > 0).sum()),
        "oos_total_tc_pct": float(100.0 * panel_oos["overlay_cost_CS"].sum()),
    }
    return out

def run_kalman_hybrid(scores_hybrid: pd.DataFrame, base_r_full: pd.Series, basket_cost_full: pd.Series):
    base_r_full = pd.Series(base_r_full).dropna().astype(float).sort_index()
    basket_cost_full = pd.Series(basket_cost_full).reindex(base_r_full.index).ffill().fillna(0.0)

    dev_idx = pd.DatetimeIndex(dev.index).intersection(scores_hybrid.index).intersection(base_r_full.index)
    oos_idx = pd.DatetimeIndex(oos.index).intersection(scores_hybrid.index).intersection(base_r_full.index)

    calib_table, best_omega, best_alpha, best_dev_panel = calibrate_hybrid_dev(
        scores=scores_hybrid,
        base_r_dev=base_r_full.reindex(dev_idx),
        basket_cost_dev=basket_cost_full.reindex(dev_idx),
        alpha_grid=KALMAN_ALPHA_GRID,
        omega_grid=KALMAN_OMEGA_GRID,
    )

    p_full = pd.Series(
        [combine_pvalues_geomean(pp, pf, best_omega) for pp, pf in zip(scores_hybrid["p_past"], scores_hybrid["p_fwd"])],
        index=scores_hybrid.index,
        name="p_value",
    )
    target_full = p_full.apply(lambda p: fast_target_from_pvalue(p, best_alpha)).rename("target_exposure")
    exp_signal_full = actual_exposure_from_targets(target_full, p_full)
    valid_idx = p_full.index.intersection(base_r_full.index)
    panel_full = compute_overlay_returns(base_r_full.reindex(valid_idx), exp_signal_full.reindex(valid_idx), basket_cost_full.reindex(valid_idx))
    panel_oos = panel_full.loc[panel_full.index.intersection(oos_idx)].copy()

    perf = pd.DataFrame({
        "PC1_base": perf_stats(panel_oos["base_r"]),
        "Fast_Kalman_hybrid_gross": perf_stats(panel_oos["overlay_r_gross"]),
        "Fast_Kalman_hybrid_net_CS": perf_stats(panel_oos["overlay_r_net_CS"]),
    }).T

    out = {
        "name": "Fast_Kalman_hybrid",
        "scores": scores_hybrid,
        "alpha_grid": KALMAN_ALPHA_GRID,
        "omega_grid": KALMAN_OMEGA_GRID,
        "calibration_table": calib_table,
        "selected_alpha": best_alpha,
        "selected_omega": best_omega,
        "dev_panel": best_dev_panel,
        "dev_objective": float(calib_table.loc[0, "Martin"]) if len(calib_table) else np.nan,
        "dev_total_tc_pct": float(calib_table.loc[0, "total_tc_pct"]) if len(calib_table) else np.nan,
        "dev_n_trades": int(calib_table.loc[0, "n_trades"]) if len(calib_table) else np.nan,
        "panel_full": panel_full,
        "panel_oos": panel_oos,
        "base_r": panel_oos["base_r"],
        "overlay_r_gross": panel_oos["overlay_r_gross"],
        "overlay_r_net": panel_oos["overlay_r_net_CS"],
        "signal_exposure_daily": panel_oos["exposure_signal"],
        "exposure_daily": panel_oos["exposure_applied"],
        "tc_daily": panel_oos["overlay_cost_CS"],
        "perf_table": perf,
        "oos_trades": int((panel_oos["overlay_turnover"] > 0).sum()),
        "oos_total_tc_pct": float(100.0 * panel_oos["overlay_cost_CS"].sum()),
    }
    return out

# -------------------------
# Run fast variants
# -------------------------
pc1_full = pd.concat([dev[BASE_COL], oos[BASE_COL]]).sort_index().drop_duplicates()
full_index = iv.index.union(pc1_full.index)
z_panel = build_kalman_iv_signals(iv, full_index)
basket_cost_full = basket_oneway_cost_series(pd.DatetimeIndex(pc1_full.index), pc_core=PC_CORE)

scores_past = kalman_local_level_scores(
    z_panel=z_panel,
    start_eval=dev.index.min(),
    init_obs=KALMAN_INIT_OBS,
    lambda_level=KALMAN_LAMBDA_LEVEL,
    cov_shrink=KALMAN_COV_SHRINK,
    roll_window=KALMAN_ROLL_WINDOW,
)

scores_fwd = var1_forward_scores(
    z_panel=z_panel,
    start_eval=dev.index.min(),
    init_obs=KALMAN_INIT_OBS,
    h_fwd=KALMAN_H_FWD,
    var_window=KALMAN_VAR_WINDOW,
    var_min_obs=KALMAN_VAR_MIN_OBS,
    cov_shrink=KALMAN_COV_SHRINK,
)

scores_hybrid = pd.concat([scores_past[["p_past", "score_sum_past", "d2_bwd"]], scores_fwd], axis=1).dropna(subset=["p_past", "p_fwd"])

fast_kalman_robust = run_kalman_robust(
    scores_past=scores_past,
    base_r_full=pc1_full,
    basket_cost_full=basket_cost_full,
)

fast_kalman_hybrid = run_kalman_hybrid(
    scores_hybrid=scores_hybrid,
    base_r_full=pc1_full,
    basket_cost_full=basket_cost_full,
)

print("Kalman fast-layer calibration (development sample)")
print("\nRobust variant (alpha only):")
display(fast_kalman_robust["calibration_table"].round(4))
print("\nHybrid variant (omega + alpha):")
display(fast_kalman_hybrid["calibration_table"].round(4))

print("\nOOS performance — robust variant")
display(fast_kalman_robust["perf_table"].round(4))
print(f"Trades: {fast_kalman_robust['oos_trades']} | Total TC: {fast_kalman_robust['oos_total_tc_pct']:.2f}%")

print("\nOOS performance — hybrid variant")
display(fast_kalman_hybrid["perf_table"].round(4))
print(f"Trades: {fast_kalman_hybrid['oos_trades']} | Total TC: {fast_kalman_hybrid['oos_total_tc_pct']:.2f}%")

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
pd.DataFrame({
    "p_past": scores_hybrid.loc[oos.index.intersection(scores_hybrid.index), "p_past"],
    "p_fwd": scores_hybrid.loc[oos.index.intersection(scores_hybrid.index), "p_fwd"],
}).plot(ax=axes[0], title="Fast-layer statistical p-values (OOS)")
axes[0].axhline(KALMAN_P_ENTER, color="orange", ls="--", lw=1.0)
axes[0].axhline(KALMAN_P_MID, color="red", ls="--", lw=1.0)
axes[0].axhline(KALMAN_P_FLOOR, color="darkred", ls="--", lw=1.0)
axes[0].set_ylabel("p-value")

pd.DataFrame({
    "robust_exp": fast_kalman_robust["exposure_daily"],
    "hybrid_exp": fast_kalman_hybrid["exposure_daily"],
}).plot(ax=axes[1], title="Kalman fast-layer applied exposures (OOS)")
axes[1].set_ylabel("Exposure")

pd.DataFrame({
    "PC1_base": (1.0 + fast_kalman_robust["base_r"]).cumprod(),
    "Fast_Kalman_robust_net": (1.0 + fast_kalman_robust["overlay_r_net"]).cumprod(),
    "Fast_Kalman_hybrid_net": (1.0 + fast_kalman_hybrid["overlay_r_net"]).cumprod(),
}).plot(ax=axes[2], title="Equity curves — base vs Kalman fast layers (net, OOS)")
axes[2].set_ylabel("Cumulative wealth")
plt.tight_layout()
plt.show()

print("\nSaved: fast_kalman_robust, fast_kalman_hybrid, scores_past, scores_fwd, scores_hybrid")


In [ ]:
# ============================================================
# V7c — Gated Kalman/Mahalanobis fast overlay V2
# ============================================================
# Purpose:
#   Two-stage fast layer:
#
#   1. Backward/current local-level Kalman-NIS opens a short-horizon gate.
#      This is deliberately short (1–3 days): it detects whether something
#      statistically unusual is happening now/recently.
#
#   2. Only when the gate is open, a VAR(1) forward module sizes exposure.
#      Forward confirmation is multi-horizon: H in a candidate set, with
#      p-values combined by the median to avoid a single noisy horizon
#      dominating the trade decision.
#
# Calibration:
#   backward window, forward horizon set, gate threshold and alpha are selected
#   on purged/embargoed development folds using a Martin_net objective penalized
#   for transaction costs. Final OOS performance is reported only after the
#   policy is fixed.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2
from IPython.display import display
import matplotlib.pyplot as plt

assert "z_panel" in globals(), "Run V7b first so that z_panel exists."
assert "pc1_full" in globals() and "basket_cost_full" in globals(), "Run V7b first."
assert "kalman_local_level_scores" in globals(), "Run V7b first."
assert "fit_var1_ols" in globals() and "inv_psd" in globals(), "Run V7b first."
assert "compute_overlay_returns" in globals(), "Run V7b first."
assert "fast_target_from_pvalue" in globals() and "actual_exposure_from_targets" in globals(), "Run V7b first."
assert "perf_stats" in globals(), "Run V7b first."
assert "dev" in globals() and "oos" in globals(), "Run the base-panel cell first."

# -------------------------
# Gated-Kalman V2 configuration
# -------------------------
# Short backward gate: detect anomaly quickly, do not wait for a 5-day accumulated signal.
GATED_BWD_WINDOW_GRID = [1, 2, 3]

# Forward confirmation: include 3d plus weekly/multi-horizon variants.
# Median p-value across horizons is used when a tuple contains multiple horizons.
GATED_FWD_HORIZON_SETS = [
    (3,),
    (5,),
    (3, 5),
    (5, 10),
    (3, 5, 10),
    (5, 10, 21),
    (3, 5, 10, 21),
]

GATED_GATE_GRID = [0.05, 0.025, 0.01]
GATED_ALPHA_GRID = KALMAN_ALPHA_GRID
GATED_LAMBDA_TC = 2.0
GATED_EMBARGO_DAYS = globals().get("PURGED_CV_EMBARGO_DAYS", 10)
GATED_N_SPLITS = globals().get("PURGED_CV_SPLITS", 4)

MAX_GATED_FWD_H = max(max(hs) for hs in GATED_FWD_HORIZON_SETS)


def _make_folds(index, n_splits=4):
    if "make_contiguous_time_folds" in globals():
        return make_contiguous_time_folds(index, n_splits=n_splits)
    idx = pd.DatetimeIndex(index).sort_values()
    parts = np.array_split(np.arange(len(idx)), n_splits)
    return [idx[p] for p in parts if len(p) > 0]


def _purge_fold(fold_idx, horizon=5, embargo_days=10):
    fold_idx = pd.DatetimeIndex(fold_idx).sort_values()
    if len(fold_idx) == 0:
        return fold_idx
    if "purged_eval_mask" in globals():
        return purged_eval_mask(fold_idx, fold_idx, horizon=horizon, embargo_days=embargo_days)
    pad = int(horizon) + int(embargo_days)
    left = fold_idx.min() + pd.tseries.offsets.BDay(pad)
    right = fold_idx.max() - pd.tseries.offsets.BDay(pad)
    keep = fold_idx[(fold_idx >= left) & (fold_idx <= right)]
    return keep if len(keep) >= max(5, horizon // 2) else fold_idx


def var1_multi_horizon_forward_scores(
    z_panel: pd.DataFrame,
    start_eval: pd.Timestamp,
    horizons,
    init_obs: int = KALMAN_INIT_OBS,
    var_window: int = KALMAN_VAR_WINDOW,
    var_min_obs: int = KALMAN_VAR_MIN_OBS,
    cov_shrink: float = KALMAN_COV_SHRINK,
) -> pd.DataFrame:
    """
    Fit one rolling VAR(1) per date and produce forward Mahalanobis p-values
    for multiple horizons. This avoids re-fitting the same VAR for each H.
    """
    horizons = sorted(set(int(h) for h in horizons))
    max_h = max(horizons)

    z_panel = z_panel.dropna().astype(float).sort_index()
    K = z_panel.shape[1]
    if len(z_panel) < max(init_obs, var_min_obs) + max_h + 1:
        raise ValueError("Signal panel too short for multi-horizon forward detector.")

    rows = []
    for loc in range(init_obs, len(z_panel)):
        dt = z_panel.index[loc]
        hist = z_panel.iloc[max(0, loc - var_window + 1):loc + 1].copy()

        row_out = {"date": dt}
        for h in horizons:
            row_out[f"score_sum_fwd_{h}"] = np.nan
            row_out[f"p_fwd_{h}"] = np.nan
            row_out[f"d2_fwd_mean_{h}"] = np.nan

        if len(hist) < var_min_obs:
            rows.append(row_out)
            continue

        fit = fit_var1_ols(hist, cov_shrink=cov_shrink)
        if fit is None:
            rows.append(row_out)
            continue

        c, A, Omega = fit
        z_curr = hist.iloc[-1].values.astype(float)
        normal_center = hist.mean(axis=0).values.astype(float)

        mu = z_curr.copy()
        Sigma = np.zeros((K, K), dtype=float)
        d2_path = []

        for step in range(1, max_h + 1):
            mu = c + A @ mu
            Sigma = A @ Sigma @ A.T + Omega
            diff = mu - normal_center
            d2 = float(diff @ inv_psd(Sigma + 1e-10 * np.eye(K)) @ diff)
            d2_path.append(d2)

            if step in horizons:
                score_sum = float(np.sum(d2_path))
                p_fwd = float(chi2.sf(score_sum, df=K * step))
                row_out[f"score_sum_fwd_{step}"] = score_sum
                row_out[f"p_fwd_{step}"] = p_fwd
                row_out[f"d2_fwd_mean_{step}"] = float(np.mean(d2_path))

        rows.append(row_out)

    out = pd.DataFrame(rows).set_index("date")
    return out.loc[out.index >= pd.Timestamp(start_eval)].copy()


def _horizon_key(horizons) -> str:
    return "/".join(str(int(h)) for h in horizons)


def gated_pvalue_series(
    scores_past: pd.DataFrame,
    scores_fwd_multi: pd.DataFrame,
    gate_p: float,
    fwd_horizons,
):
    """
    Gate closed -> p=1 -> no fast de-risking.
    Gate open   -> median of selected forward p-values drives sizing.
    """
    fwd_cols = [f"p_fwd_{int(h)}" for h in fwd_horizons]
    missing = [c for c in fwd_cols if c not in scores_fwd_multi.columns]
    if missing:
        raise ValueError(f"Missing forward p-value columns: {missing}")

    scores = pd.concat(
        [scores_past[["p_past"]], scores_fwd_multi[fwd_cols]],
        axis=1
    ).sort_index()

    p_fwd_median = scores[fwd_cols].median(axis=1, skipna=True).rename("p_fwd_median")
    n_fwd_valid = scores[fwd_cols].notna().sum(axis=1).rename("n_fwd_valid")

    p_eff = pd.Series(1.0, index=scores.index, name="p_value")
    gate_open = (scores["p_past"] <= float(gate_p)) & p_fwd_median.notna()
    p_eff.loc[gate_open] = p_fwd_median.loc[gate_open].clip(lower=1e-12, upper=1.0)

    diagnostics = pd.DataFrame({
        "p_past": scores["p_past"],
        "p_fwd_median": p_fwd_median,
        "n_fwd_valid": n_fwd_valid,
        "gate_open": gate_open,
        "p_value": p_eff,
    }, index=scores.index)

    return p_eff, gate_open.rename("gate_open"), diagnostics


def gated_panel_for_params(
    scores_past: pd.DataFrame,
    scores_fwd_multi: pd.DataFrame,
    gate_p: float,
    alpha: float,
    fwd_horizons,
    base_r: pd.Series,
    basket_cost: pd.Series,
):
    p_eff, gate_open, diag = gated_pvalue_series(
        scores_past=scores_past,
        scores_fwd_multi=scores_fwd_multi,
        gate_p=gate_p,
        fwd_horizons=fwd_horizons,
    )

    target = p_eff.apply(lambda p: fast_target_from_pvalue(p, alpha)).rename("target_exposure")
    exp_signal = actual_exposure_from_targets(target, p_eff)

    valid_idx = pd.DatetimeIndex(base_r.index).intersection(exp_signal.index).intersection(basket_cost.index)
    panel = compute_overlay_returns(
        base_r.reindex(valid_idx),
        exp_signal.reindex(valid_idx),
        basket_cost.reindex(valid_idx),
    )

    panel["p_value"] = p_eff.reindex(panel.index)
    panel["gate_open"] = gate_open.reindex(panel.index).fillna(False).astype(bool)
    panel["p_past"] = diag["p_past"].reindex(panel.index)
    panel["p_fwd_median"] = diag["p_fwd_median"].reindex(panel.index)
    panel["n_fwd_valid"] = diag["n_fwd_valid"].reindex(panel.index)
    return panel


def gated_cv_score(panel_full: pd.DataFrame, dev_index: pd.DatetimeIndex, purge_horizon: int):
    dev_index = pd.DatetimeIndex(dev_index).intersection(panel_full.index).sort_values()
    folds = _make_folds(dev_index, n_splits=GATED_N_SPLITS)
    rows = []

    for j, fold in enumerate(folds, start=1):
        val_idx = _purge_fold(fold, horizon=purge_horizon, embargo_days=GATED_EMBARGO_DAYS)
        val_idx = pd.DatetimeIndex(val_idx).intersection(panel_full.index)
        if len(val_idx) < 5:
            continue

        sub = panel_full.loc[val_idx]
        ps = perf_stats(sub["overlay_r_net_CS"])
        tc_pct = float(100.0 * sub["overlay_cost_CS"].sum())
        utility = ps["Martin"]
        if np.isfinite(utility):
            utility -= GATED_LAMBDA_TC * tc_pct

        rows.append({
            "fold": j,
            "n": len(sub),
            "Martin": ps["Martin"],
            "Sharpe": ps["Sharpe"],
            "tc_pct": tc_pct,
            "utility": utility,
        })

    if not rows:
        return {"cv_utility": -np.inf, "cv_Martin": np.nan, "cv_Sharpe": np.nan, "cv_tc_pct": np.nan, "n_folds": 0}

    df = pd.DataFrame(rows)
    return {
        "cv_utility": float(df["utility"].mean()),
        "cv_Martin": float(df["Martin"].mean()),
        "cv_Sharpe": float(df["Sharpe"].mean()),
        "cv_tc_pct": float(df["tc_pct"].mean()),
        "n_folds": int(len(df)),
    }


def calibrate_gated_kalman_v2(base_r_full: pd.Series, basket_cost_full: pd.Series):
    rows = []
    panels = {}
    dev_idx = pd.DatetimeIndex(dev.index).intersection(base_r_full.index)

    # Cache past scores for short backward gates.
    scores_past_cache = {}
    for bw in GATED_BWD_WINDOW_GRID:
        scores_past_cache[int(bw)] = kalman_local_level_scores(
            z_panel=z_panel,
            start_eval=dev.index.min(),
            init_obs=KALMAN_INIT_OBS,
            lambda_level=KALMAN_LAMBDA_LEVEL,
            cov_shrink=KALMAN_COV_SHRINK,
            roll_window=int(bw),
        )

    # Compute all forward horizons once; selected horizon sets use median p-values.
    all_horizons = sorted({h for hs in GATED_FWD_HORIZON_SETS for h in hs})
    scores_fwd_multi = var1_multi_horizon_forward_scores(
        z_panel=z_panel,
        start_eval=dev.index.min(),
        horizons=all_horizons,
        init_obs=KALMAN_INIT_OBS,
        var_window=KALMAN_VAR_WINDOW,
        var_min_obs=KALMAN_VAR_MIN_OBS,
        cov_shrink=KALMAN_COV_SHRINK,
    )

    for bw in GATED_BWD_WINDOW_GRID:
        scores_past_i = scores_past_cache[int(bw)]

        for fwd_hs in GATED_FWD_HORIZON_SETS:
            fwd_hs = tuple(int(h) for h in fwd_hs)
            fwd_key = _horizon_key(fwd_hs)
            purge_h = max(int(bw), max(fwd_hs))

            for gate_p in GATED_GATE_GRID:
                for alpha in GATED_ALPHA_GRID:
                    panel = gated_panel_for_params(
                        scores_past=scores_past_i,
                        scores_fwd_multi=scores_fwd_multi,
                        gate_p=gate_p,
                        alpha=alpha,
                        fwd_horizons=fwd_hs,
                        base_r=base_r_full,
                        basket_cost=basket_cost_full,
                    )

                    cv = gated_cv_score(panel, dev_idx, purge_horizon=purge_h)
                    panel_dev = panel.loc[panel.index.intersection(dev_idx)].copy()
                    ps_dev = perf_stats(panel_dev["overlay_r_net_CS"])
                    dev_tc = float(100.0 * panel_dev["overlay_cost_CS"].sum())

                    rows.append({
                        "bwd_window": int(bw),
                        "fwd_horizons": fwd_key,
                        "gate_p": float(gate_p),
                        "alpha": float(alpha),
                        "dev_Martin": ps_dev["Martin"],
                        "dev_Sharpe": ps_dev["Sharpe"],
                        "dev_total_tc_pct": dev_tc,
                        "dev_n_trades": int((panel_dev["overlay_turnover"] > 0).sum()),
                        "dev_gate_open_days": int(panel_dev["gate_open"].sum()) if "gate_open" in panel_dev.columns else np.nan,
                        "purge_horizon": int(purge_h),
                        **cv,
                    })

                    panels[(int(bw), fwd_key, float(gate_p), float(alpha))] = panel

    table = (
        pd.DataFrame(rows)
        .sort_values(["cv_utility", "cv_Martin", "dev_total_tc_pct"], ascending=[False, False, True])
        .reset_index(drop=True)
    )

    best = table.iloc[0]
    key = (int(best["bwd_window"]), str(best["fwd_horizons"]), float(best["gate_p"]), float(best["alpha"]))
    return table, key, panels[key]


gated_calib_table, gated_selected_key, gated_panel_full = calibrate_gated_kalman_v2(pc1_full, basket_cost_full)

gated_bwd_window, gated_fwd_horizons, gated_gate_p, gated_alpha = gated_selected_key
gated_oos = gated_panel_full.loc[gated_panel_full.index.intersection(oos.index)].copy()

fast_kalman_gated = {
    "name": "Fast_Kalman_gated",
    "calibration_table": gated_calib_table,
    "selected_bwd_window": gated_bwd_window,
    "selected_fwd_horizons": gated_fwd_horizons,
    "selected_gate_p": gated_gate_p,
    "selected_alpha": gated_alpha,
    "dev_objective": float(gated_calib_table.loc[0, "cv_Martin"]),
    "dev_total_tc_pct": float(gated_calib_table.loc[0, "dev_total_tc_pct"]),
    "dev_n_trades": int(gated_calib_table.loc[0, "dev_n_trades"]),
    "panel_full": gated_panel_full,
    "panel_oos": gated_oos,
    "base_r": gated_oos["base_r"],
    "overlay_r_gross": gated_oos["overlay_r_gross"],
    "overlay_r_net": gated_oos["overlay_r_net_CS"],
    "signal_exposure_daily": gated_oos["exposure_signal"],
    "exposure_daily": gated_oos["exposure_applied"],
    "tc_daily": gated_oos["overlay_cost_CS"],
    "oos_trades": int((gated_oos["overlay_turnover"] > 0).sum()),
    "oos_total_tc_pct": float(100.0 * gated_oos["overlay_cost_CS"].sum()),
    "perf_table": pd.DataFrame({
        "PC1_base": perf_stats(gated_oos["base_r"]),
        "Fast_Kalman_gated_gross": perf_stats(gated_oos["overlay_r_gross"]),
        "Fast_Kalman_gated_net_CS": perf_stats(gated_oos["overlay_r_net_CS"]),
    }).T,
}

print("Gated Kalman V2 fast-layer calibration — short backward gate + multi-horizon median forward p-value")
display(gated_calib_table.round(4).head(15))
print(
    f"\nSelected gated Kalman V2: "
    f"bwd_window={gated_bwd_window}d, "
    f"fwd_horizons={gated_fwd_horizons}, "
    f"gate_p={gated_gate_p:.4%}, "
    f"alpha={gated_alpha:.2f}"
)
print("\nOOS performance — gated Kalman V2")
display(fast_kalman_gated["perf_table"].round(4))
print(f"Trades: {fast_kalman_gated['oos_trades']} | Total TC: {fast_kalman_gated['oos_total_tc_pct']:.2f}%")

print("\nSaved: fast_kalman_gated, gated_calib_table")


## 7) Final validation: decision rules, benchmarks, subperiods, and bootstrap inference

In [ ]:
# ============================================================
# V8 — Final overlay validation table, deterministic benchmarks,
#      subperiod analysis, combined decision rules, and bootstrap inference
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

assert "oos" in globals() and "dev" in globals(), "Run the base-panel cell first."
assert "W_hist_long" in globals() and "asset_costs" in globals(), "Run the walk-forward PCA and CS cost blocks first."
assert "px_stocks" in globals(), "Run the data section first."

ANN = 252
PC_CORE = globals().get("PC_CORE", "PC1")
BASE_COL = globals().get("BASE_COL", "pc1_base")
BOOT_B_FINAL = 2000
BLOCK_LEN = int(globals().get("BOOTSTRAP_BLOCK_LEN", 30))
BLOCK_METHOD = globals().get("BOOTSTRAP_BLOCK_METHOD", "fallback fixed value")

# ------------------------------------------------------------
# Core helpers
# ------------------------------------------------------------
def perf_full(return_dict, ann=ANN):
    rows = []
    for name, r in return_dict.items():
        stats = perf_stats(pd.Series(r).dropna().astype(float), ann=ann)
        rows.append({"strategy": name, **stats})
    return pd.DataFrame(rows).set_index("strategy")

def compute_overlay_panel_from_exposure(base_r: pd.Series, exposure_signal: pd.Series, basket_cost: pd.Series):
    idx = pd.DatetimeIndex(base_r.index).intersection(exposure_signal.index).intersection(basket_cost.index)
    base = pd.Series(base_r).reindex(idx).astype(float)
    exp_signal = pd.Series(exposure_signal).reindex(idx).fillna(1.0).astype(float).rename("exposure_signal")
    exp_applied = exp_signal.shift(1).fillna(1.0).rename("exposure_applied")
    turnover = exp_applied.diff().abs().fillna(0.0).rename("overlay_turnover")
    tc = (turnover * basket_cost.reindex(idx).fillna(0.0)).rename("overlay_cost_CS")
    gross = (exp_applied * base).rename("overlay_r_gross")
    net = (gross - tc).rename("overlay_r_net_CS")
    return pd.concat(
        [base.rename("base_r"), exp_signal, exp_applied, turnover, basket_cost.reindex(idx), tc, gross, net],
        axis=1
    )

def stationary_bootstrap_indices(n: int, mean_block: int = 30, n_boot: int = 1000, seed: int = 42):
    rng = np.random.default_rng(seed)
    p = 1.0 / max(mean_block, 1)
    out = np.empty((n_boot, n), dtype=int)
    for b in range(n_boot):
        idx = np.empty(n, dtype=int)
        idx[0] = rng.integers(0, n)
        for t in range(1, n):
            if rng.random() < p:
                idx[t] = rng.integers(0, n)
            else:
                idx[t] = (idx[t-1] + 1) % n
        out[b] = idx
    return out

def bootstrap_compare(base: pd.Series, target: pd.Series, label: str, n_boot: int = BOOT_B_FINAL, block_len: int = BLOCK_LEN, seed: int = 42):
    both = pd.concat([pd.Series(base, name="base"), pd.Series(target, name="target")], axis=1).dropna()
    idx_boot = stationary_bootstrap_indices(len(both), mean_block=block_len, n_boot=n_boot, seed=seed)
    rows = []
    arr = both.values
    for idx in idx_boot:
        b = arr[idx, 0]
        t = arr[idx, 1]
        rows.append({
            "Delta Sharpe": perf_stats(pd.Series(t))["Sharpe"] - perf_stats(pd.Series(b))["Sharpe"],
            "Delta MaxDD": abs(perf_stats(pd.Series(b))["MaxDD"]) - abs(perf_stats(pd.Series(t))["MaxDD"]),
        })
    df = pd.DataFrame(rows)
    summ = pd.DataFrame({
        "mean": df.mean(),
        "p05": df.quantile(0.05),
        "p50": df.quantile(0.50),
        "p95": df.quantile(0.95),
        "P(target_wins)": [
            float((df["Delta Sharpe"] > 0).mean()),
            float((df["Delta MaxDD"] > 0).mean()),
        ],
    }, index=["Delta Sharpe", "Delta MaxDD"])
    print(f"\nBootstrap OOS uncertainty ({label}; N={n_boot}, mean block={block_len}d, method={BLOCK_METHOD}):")
    display(summ.round(4))
    return df, summ

def materialize_panel(obj, base_r: pd.Series, basket_cost: pd.Series):
    required_cols = {
        "base_r", "exposure_signal", "exposure_applied",
        "overlay_turnover", "overlay_cost_CS",
        "overlay_r_gross", "overlay_r_net_CS"
    }

    if isinstance(obj, pd.DataFrame):
        if required_cols.issubset(set(obj.columns)):
            panel = obj.copy()
            idx = pd.DatetimeIndex(panel.index).intersection(base_r.index).intersection(basket_cost.index)
            return panel.reindex(idx)
        if "exposure_signal" in obj.columns:
            return compute_overlay_panel_from_exposure(base_r, obj["exposure_signal"], basket_cost)
        if "exposure_applied" in obj.columns:
            return compute_overlay_panel_from_exposure(base_r, obj["exposure_applied"], basket_cost)
        raise ValueError("Unsupported DataFrame format for overlay panel.")

    if isinstance(obj, dict):
        if "panel_full" in obj and isinstance(obj["panel_full"], pd.DataFrame):
            panel = obj["panel_full"].copy()
            if required_cols.issubset(set(panel.columns)):
                idx = pd.DatetimeIndex(panel.index).intersection(base_r.index).intersection(basket_cost.index)
                return panel.reindex(idx)
        if "panel_oos" in obj and isinstance(obj["panel_oos"], pd.DataFrame):
            panel = obj["panel_oos"].copy()
            if required_cols.issubset(set(panel.columns)):
                idx = pd.DatetimeIndex(panel.index).intersection(base_r.index).intersection(basket_cost.index)
                return panel.reindex(idx)
        if "signal_exposure_daily" in obj:
            return compute_overlay_panel_from_exposure(base_r, obj["signal_exposure_daily"], basket_cost)
        if "exposure_daily" in obj:
            return compute_overlay_panel_from_exposure(base_r, obj["exposure_daily"], basket_cost)
        raise ValueError("Unsupported dict format for overlay panel.")

    if isinstance(obj, pd.Series):
        return compute_overlay_panel_from_exposure(base_r, obj, basket_cost)

    raise ValueError("Unsupported overlay object type.")

# NEW: robust helpers for selected slow-layer rebuild
def _call_state_builder(builder, r: pd.Series):
    """
    Some notebooks define state_builders[state_name] as lambda r: ...
    others as zero-arg lambdas. Support both.
    """
    try:
        return builder(r)
    except TypeError:
        return builder()

def _get_selected_tau(attrs: dict):
    if "selected_tau" in attrs:
        return float(attrs["selected_tau"])
    if "selected_tau_q" in attrs:
        return float(attrs["selected_tau_q"])
    raise KeyError("Neither 'selected_tau' nor 'selected_tau_q' found in attrs.")

# ------------------------------------------------------------
# Align full-sample base and costs
# ------------------------------------------------------------
base_r_full = pd.concat([dev[BASE_COL], oos[BASE_COL]]).sort_index().drop_duplicates().astype(float)
basket_cost_full = basket_oneway_cost_series(pd.DatetimeIndex(base_r_full.index), pc_core=PC_CORE)

# ------------------------------------------------------------
# Selected slow layer (rebuild on full sample to support dev calibration)
# ------------------------------------------------------------
slow_selected_panel_full = None
slow_selected_label = None
slow_selected_meta = None

if "ou_cv_selected" in globals() and isinstance(ou_cv_selected, pd.DataFrame):
    attrs = getattr(ou_cv_selected, "attrs", {})
    if all(k in attrs for k in ["selected_state_var", "selected_H"]):
        assert "state_builders" in globals() and "make_ou_scores" in globals() and "build_weekly_overlay_from_scores" in globals(), \
            "Run V6b/V6c first so that OU helpers exist."
        state_name = attrs["selected_state_var"]
        H_best = int(attrs["selected_H"])
        tau = _get_selected_tau(attrs)
        state_full = _call_state_builder(state_builders[state_name], base_r_full)
        score_name = f"cv_selected_{state_name}_H{H_best}_full"
        scores_full = make_ou_scores(base_r_full.index, state_full, H_best, tau, score_name)
        exp_sig_full, exp_app_full, overlay_full, trades_full = build_weekly_overlay_from_scores(scores_full, base_r_full)
        slow_selected_panel_full = compute_overlay_panel_from_exposure(base_r_full, exp_sig_full, basket_cost_full)
        slow_selected_label = "Slow_OU_selected_CV"
        slow_selected_meta = {
            "selection_rule": attrs.get("selection_rule", "purged/embargoed CV"),
            "selection_state_var": state_name,
            "selection_H": H_best,
            "selection_tau_q": attrs.get("selected_tau_q", np.nan),
        }

if slow_selected_panel_full is None and "ou_sweep_selected" in globals() and isinstance(ou_sweep_selected, pd.DataFrame):
    attrs = getattr(ou_sweep_selected, "attrs", {})
    if all(k in attrs for k in ["selected_state_var", "selected_H"]):
        assert "state_builders" in globals() and "make_ou_scores" in globals() and "build_weekly_overlay_from_scores" in globals(), \
            "Run V6b first so that OU helpers exist."
        state_name = attrs["selected_state_var"]
        H_best = int(attrs["selected_H"])
        tau = _get_selected_tau(attrs)
        state_full = _call_state_builder(state_builders[state_name], base_r_full)
        score_name = f"sweep_selected_{state_name}_H{H_best}_full"
        scores_full = make_ou_scores(base_r_full.index, state_full, H_best, tau, score_name)
        exp_sig_full, exp_app_full, overlay_full, trades_full = build_weekly_overlay_from_scores(scores_full, base_r_full)
        slow_selected_panel_full = compute_overlay_panel_from_exposure(base_r_full, exp_sig_full, basket_cost_full)
        slow_selected_label = "Slow_OU_selected_sweep"
        slow_selected_meta = {
            "selection_rule": attrs.get("selection_rule", "development-only AUC"),
            "selection_state_var": state_name,
            "selection_H": H_best,
            "selection_tau_q": attrs.get("selected_tau_q", np.nan),
        }

if slow_selected_panel_full is None:
    assert "ou_slow_overlay" in globals(), "Run V6 first."
    slow_selected_panel_full = materialize_panel(ou_slow_overlay, base_r_full, basket_cost_full)
    slow_selected_label = "Slow_OU_prespecified"
    slow_selected_meta = {"selection_rule": "pre-specified"}

print("Selected slow layer:")
display(pd.DataFrame([slow_selected_meta]).T.rename(columns={0: "value"}))

# ------------------------------------------------------------
# Medium HAR layer rebuilt on full sample (same logic as V5.1/V5.2)
# ------------------------------------------------------------
assert "har_res" in globals() and "har_resid" in globals(), "Run V5 first so that HAR model objects exist."
assert "HAR_COLS" in globals() and "Y_COL" in globals(), "Run V5 first."
tau_har = float(globals().get("tau", dev[Y_COL].dropna().quantile(globals().get("TAU_PERCENTILE", 0.80))))
HAR_BOOT_B_SMALL = 1000
rng_har = np.random.default_rng(123)
resid_pool = np.asarray(har_resid.dropna().values, dtype=float)

full_har = pd.concat([dev[[Y_COL] + HAR_COLS], oos[[Y_COL] + HAR_COLS]], axis=0).sort_index()
full_har = full_har[~full_har.index.duplicated(keep="last")]
full_har = full_har.dropna().copy()

X_full_har = sm.add_constant(full_har[HAR_COLS], has_constant="add")
mu_full_har = pd.Series(har_res.predict(X_full_har), index=full_har.index, name="mu_hat")

boot_draws = rng_har.choice(resid_pool, size=(len(mu_full_har), HAR_BOOT_B_SMALL), replace=True)
pred_draws = np.clip(mu_full_har.values.reshape(-1, 1) + boot_draws, 0.0, None)
p_rv_exceed_full = pd.Series((pred_draws > tau_har).mean(axis=1), index=mu_full_har.index, name="p_rv_exceed")

score_daily_full = p_rv_exceed_full.reindex(base_r_full.index).dropna()
score_weekly = score_daily_full.resample("W-FRI").last().dropna()

EXP_NORMAL = 1.00
EXP_DEF = 0.75
EXP_EXT = 0.50
ENTER_DEF_Q = 0.90
EXIT_DEF_Q = 0.75
ENTER_EXT_Q = 0.98
EXIT_EXT_Q = 0.90
MIN_HIST_WEEKS = 26

state_rows = []
state = "normal"
for i, dt in enumerate(score_weekly.index):
    p_now = float(score_weekly.loc[dt])
    hist = score_weekly.iloc[:i]
    if len(hist) < MIN_HIST_WEEKS:
        state = "normal"
        exposure = EXP_NORMAL
    else:
        q_def_enter = float(hist.quantile(ENTER_DEF_Q))
        q_def_exit  = float(hist.quantile(EXIT_DEF_Q))
        q_ext_enter = float(hist.quantile(ENTER_EXT_Q))
        q_ext_exit  = float(hist.quantile(EXIT_EXT_Q))

        if state == "normal":
            if p_now >= q_ext_enter:
                state = "extreme"
            elif p_now >= q_def_enter:
                state = "defensive"
        elif state == "defensive":
            if p_now >= q_ext_enter:
                state = "extreme"
            elif p_now <= q_def_exit:
                state = "normal"
        elif state == "extreme":
            if p_now <= q_ext_exit:
                if p_now <= q_def_exit:
                    state = "normal"
                else:
                    state = "defensive"

        exposure = {"normal": EXP_NORMAL, "defensive": EXP_DEF, "extreme": EXP_EXT}[state]

    state_rows.append({
        "Date": dt,
        "p_score": p_now,
        "state": state,
        "exposure_signal": exposure,
    })

weekly_overlay_full = pd.DataFrame(state_rows).set_index("Date")
medium_signal_full = weekly_overlay_full["exposure_signal"].reindex(base_r_full.index, method="ffill").fillna(EXP_NORMAL)
medium_har_panel_full = compute_overlay_panel_from_exposure(base_r_full, medium_signal_full, basket_cost_full)

# ------------------------------------------------------------
# Fast layers and selection
# ------------------------------------------------------------
fast_candidates = []

if "fast_kalman_robust" in globals() and isinstance(fast_kalman_robust, dict):
    fast_candidates.append(("Fast_Kalman_robust", fast_kalman_robust))

if "fast_kalman_hybrid" in globals() and isinstance(fast_kalman_hybrid, dict):
    fast_candidates.append(("Fast_Kalman_hybrid", fast_kalman_hybrid))

if "fast_kalman_gated" in globals() and isinstance(fast_kalman_gated, dict):
    fast_candidates.append(("Fast_Kalman_gated", fast_kalman_gated))

if "fast_overlay" in globals() and isinstance(fast_overlay, pd.DataFrame):
    fast_iv_full_panel = materialize_panel(fast_overlay, base_r_full, basket_cost_full)
    fast_candidates.append(("Fast_IV", {
        "name": "Fast_IV",
        "dev_objective": np.nan,
        "dev_total_tc_pct": np.nan,
        "dev_n_trades": np.nan,
        "oos_trades": int((fast_iv_full_panel.loc[oos.index.intersection(fast_iv_full_panel.index), "overlay_turnover"] > 0).sum()),
        "oos_total_tc_pct": float(100.0 * fast_iv_full_panel.loc[oos.index.intersection(fast_iv_full_panel.index), "overlay_cost_CS"].sum()),
        "panel_full": fast_iv_full_panel,
        "panel_oos": fast_iv_full_panel.loc[oos.index.intersection(fast_iv_full_panel.index)].copy(),
        "overlay_r_net": fast_iv_full_panel.loc[oos.index.intersection(fast_iv_full_panel.index), "overlay_r_net_CS"],
        "signal_exposure_daily": fast_iv_full_panel.loc[oos.index.intersection(fast_iv_full_panel.index), "exposure_signal"],
        "exposure_daily": fast_iv_full_panel.loc[oos.index.intersection(fast_iv_full_panel.index), "exposure_applied"],
        "perf_table": perf_full({
            "PC1_base": fast_iv_full_panel.loc[oos.index.intersection(fast_iv_full_panel.index), "base_r"],
            "Fast_IV_gross": fast_iv_full_panel.loc[oos.index.intersection(fast_iv_full_panel.index), "overlay_r_gross"],
            "Fast_IV_net_CS": fast_iv_full_panel.loc[oos.index.intersection(fast_iv_full_panel.index), "overlay_r_net_CS"],
        }),
    }))

assert len(fast_candidates) > 0, "No fast-layer candidates available. Run V7 or V7b first."

FAST_DEV_TC_MAX = 0.60
FAST_DEV_TRADES_MAX = 120
FAST_LAMBDA_TC = 5.0
FAST_LAMBDA_TRADES = 0.01

def _extract_fast_metrics(label, cand):
    dev_obj = cand.get("dev_objective", np.nan)
    dev_tc = cand.get("dev_total_tc_pct", np.nan)
    dev_n_trades = cand.get("dev_n_trades", np.nan)
    if not np.isfinite(dev_n_trades):
        dev_panel = cand.get("dev_panel", None)
        if isinstance(dev_panel, pd.DataFrame) and "overlay_turnover" in dev_panel.columns:
            dev_n_trades = int((dev_panel["overlay_turnover"] > 0).sum())

    oos_trades = cand.get("oos_trades", np.nan)
    oos_tc = cand.get("oos_total_tc_pct", np.nan)

    if np.isfinite(dev_obj):
        utility = float(dev_obj)
        if np.isfinite(dev_tc):
            utility -= FAST_LAMBDA_TC * float(dev_tc)
        if np.isfinite(dev_n_trades):
            utility -= FAST_LAMBDA_TRADES * float(dev_n_trades)
    else:
        utility = -np.inf

    admissible = True
    if np.isfinite(dev_tc) and dev_tc > FAST_DEV_TC_MAX:
        admissible = False
    if np.isfinite(dev_n_trades) and dev_n_trades > FAST_DEV_TRADES_MAX:
        admissible = False

    return {
        "label": label,
        "dev_objective_Martin": dev_obj,
        "dev_total_tc_pct": dev_tc,
        "dev_n_trades": dev_n_trades,
        "utility": utility,
        "admissible": admissible,
        "oos_trades": oos_trades,
        "oos_total_tc_pct": oos_tc,
    }

fast_diag = pd.DataFrame([_extract_fast_metrics(lbl, cand) for lbl, cand in fast_candidates]).set_index("label")
admissible_idx = fast_diag.index[fast_diag["admissible"] & np.isfinite(fast_diag["utility"])]
if len(admissible_idx) > 0:
    fast_main_label = fast_diag.loc[admissible_idx, "utility"].idxmax()
else:
    valid_idx = fast_diag.index[np.isfinite(fast_diag["utility"])]
    if len(valid_idx) > 0:
        fast_main_label = fast_diag.loc[valid_idx, "utility"].idxmax()
    else:
        labels = [lbl for lbl, _ in fast_candidates]
        fast_main_label = "Fast_IV" if "Fast_IV" in labels else labels[0]

fast_main = dict(fast_candidates)[fast_main_label]
fast_main_panel_full = materialize_panel(fast_main, base_r_full, basket_cost_full)

print(f"Fast layer used in combined overlay candidates: {fast_main_label}")
print("\nFast-layer selection diagnostics:")
display(fast_diag.round(4))

# ------------------------------------------------------------
# Deterministic benchmarks
# ------------------------------------------------------------
def fixed_vol_target_panel(base_r: pd.Series, basket_cost: pd.Series, target_vol: float, lookback: int = 21, max_leverage: float = 1.0):
    rolling_vol = base_r.rolling(lookback, min_periods=max(5, lookback // 2)).std(ddof=1) * np.sqrt(ANN)
    target_exp = (target_vol / rolling_vol).clip(lower=0.0, upper=max_leverage).fillna(1.0)
    target_exp.name = f"target_vol_{int(100*target_vol)}"
    return compute_overlay_panel_from_exposure(base_r, target_exp, basket_cost)

def drawdown_control_panel(base_r: pd.Series, basket_cost: pd.Series, dd_enter: float = -0.10, dd_exit: float = -0.05, exp_def: float = 0.50):
    wealth = (1.0 + pd.Series(base_r)).cumprod()
    dd = wealth / wealth.cummax() - 1.0
    state = "normal"
    exp_signal = pd.Series(index=base_r.index, dtype=float)
    for dt in base_r.index:
        dd_now = float(dd.loc[dt])
        if state == "normal" and dd_now <= dd_enter:
            state = "def"
        elif state == "def" and dd_now >= dd_exit:
            state = "normal"
        exp_signal.loc[dt] = exp_def if state == "def" else 1.0
    exp_signal.name = "drawdown_control_signal"
    return compute_overlay_panel_from_exposure(base_r, exp_signal, basket_cost)

vol_target_15_panel_full = fixed_vol_target_panel(base_r_full, basket_cost_full, target_vol=0.15)
vol_target_20_panel_full = fixed_vol_target_panel(base_r_full, basket_cost_full, target_vol=0.20)
dd_control_panel_full = drawdown_control_panel(base_r_full, basket_cost_full, dd_enter=-0.10, dd_exit=-0.05, exp_def=0.50)

# ------------------------------------------------------------
# Combined decision rules
# ------------------------------------------------------------
def combine_deficit_signals(signal_df: pd.DataFrame, weights: dict) -> pd.Series:
    df = signal_df.copy().astype(float)
    deficits = 1.0 - df
    total = 0.0
    for k, w in weights.items():
        total = total + float(w) * deficits[k]
    exp_signal = 1.0 - total
    return exp_signal.clip(lower=0.0, upper=1.0).rename("combined_signal")

def combine_min_signal(signal_df: pd.DataFrame) -> pd.Series:
    return signal_df.min(axis=1).rename("combined_signal")

layer_signals_full = pd.concat([
    slow_selected_panel_full["exposure_signal"].rename("slow"),
    medium_har_panel_full["exposure_signal"].rename("medium"),
    fast_main_panel_full["exposure_signal"].rename("fast"),
], axis=1).dropna()

COMBINED_FIXED_WEIGHTS = {"slow": 0.60, "medium": 0.25, "fast": 0.15}
combined_fixed_signal_full = combine_deficit_signals(layer_signals_full, COMBINED_FIXED_WEIGHTS)
combined_min_signal_full = combine_min_signal(layer_signals_full)

combined_fixed_panel_full = compute_overlay_panel_from_exposure(base_r_full, combined_fixed_signal_full, basket_cost_full)
combined_min_panel_full = compute_overlay_panel_from_exposure(base_r_full, combined_min_signal_full, basket_cost_full)

COMBINED_LAMBDA_TC = 2.0
COMBINED_PURGE_H = int(max(globals().get("KALMAN_ROLL_WINDOW", 5), globals().get("KALMAN_H_FWD", 2), slow_selected_meta.get("selection_H", 21) if isinstance(slow_selected_meta, dict) else 21))
COMBINED_EMBARGO_DAYS = int(globals().get("PURGED_CV_EMBARGO_DAYS", 10))
COMBINED_N_SPLITS = int(globals().get("PURGED_CV_SPLITS", 4))

def _combined_make_folds(index, n_splits=4):
    if "make_contiguous_time_folds" in globals():
        return make_contiguous_time_folds(index, n_splits=n_splits)
    idx = pd.DatetimeIndex(index).sort_values()
    parts = np.array_split(np.arange(len(idx)), n_splits)
    return [idx[p] for p in parts if len(p) > 0]

def _combined_purged_fold(fold_idx, horizon=21, embargo_days=10):
    fold_idx = pd.DatetimeIndex(fold_idx).sort_values()
    if len(fold_idx) == 0:
        return fold_idx
    if "purged_eval_mask" in globals():
        return purged_eval_mask(fold_idx, fold_idx, horizon=horizon, embargo_days=embargo_days)
    pad = int(horizon) + int(embargo_days)
    left = fold_idx.min() + pd.tseries.offsets.BDay(pad)
    right = fold_idx.max() - pd.tseries.offsets.BDay(pad)
    keep = fold_idx[(fold_idx >= left) & (fold_idx <= right)]
    return keep if len(keep) >= max(5, horizon // 2) else fold_idx

def combined_purged_cv_score(panel_full: pd.DataFrame, dev_index: pd.DatetimeIndex):
    dev_index = pd.DatetimeIndex(dev_index).intersection(panel_full.index).sort_values()
    folds = _combined_make_folds(dev_index, n_splits=COMBINED_N_SPLITS)
    rows = []
    for j, fold in enumerate(folds, start=1):
        val_idx = _combined_purged_fold(fold, horizon=COMBINED_PURGE_H, embargo_days=COMBINED_EMBARGO_DAYS)
        val_idx = pd.DatetimeIndex(val_idx).intersection(panel_full.index)
        if len(val_idx) < 5:
            continue
        sub = panel_full.loc[val_idx]
        ps = perf_stats(sub["overlay_r_net_CS"])
        tc_pct = float(100.0 * sub["overlay_cost_CS"].sum())
        utility = ps["Martin"]
        if np.isfinite(utility):
            utility -= COMBINED_LAMBDA_TC * tc_pct
        rows.append({"fold": j, "n": len(sub), "Martin": ps["Martin"], "Sharpe": ps["Sharpe"], "tc_pct": tc_pct, "utility": utility})
    if not rows:
        return {"cv_utility": -np.inf, "cv_Martin": np.nan, "cv_Sharpe": np.nan, "cv_tc_pct": np.nan, "cv_folds": 0}
    df = pd.DataFrame(rows)
    return {
        "cv_utility": float(df["utility"].mean()),
        "cv_Martin": float(df["Martin"].mean()),
        "cv_Sharpe": float(df["Sharpe"].mean()),
        "cv_tc_pct": float(df["tc_pct"].mean()),
        "cv_folds": int(len(df)),
    }

weight_grid = []
for ws in [0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    for wm in [0.15, 0.20, 0.25, 0.30, 0.35]:
        wf = round(1.0 - ws - wm, 10)
        if wf < 0:
            continue
        if not (ws >= wm >= wf):
            continue
        if wf > 0.5 * ws:
            continue
        weight_grid.append((round(ws, 2), round(wm, 2), round(wf, 2)))

dev_idx_full = base_r_full.index.intersection(dev.index)
rows = []
weight_panels = {}
for ws, wm, wf in weight_grid:
    weights = {"slow": ws, "medium": wm, "fast": wf}
    sig = combine_deficit_signals(layer_signals_full, weights)
    panel = compute_overlay_panel_from_exposure(base_r_full, sig, basket_cost_full)
    panel_dev = panel.loc[panel.index.intersection(dev_idx_full)].copy()
    ps = perf_stats(panel_dev["overlay_r_net_CS"])
    dev_tc = float(100.0 * panel_dev["overlay_cost_CS"].sum())
    dev_utility = ps["Martin"]
    if np.isfinite(dev_utility):
        dev_utility -= COMBINED_LAMBDA_TC * dev_tc
    cv = combined_purged_cv_score(panel, dev_idx_full)
    rows.append({
        "w_slow": ws,
        "w_medium": wm,
        "w_fast": wf,
        "dev_total_tc_pct": dev_tc,
        "dev_trades": int((panel_dev["overlay_turnover"] > 0).sum()),
        "dev_utility": dev_utility,
        **cv,
        **ps,
    })
    weight_panels[(ws, wm, wf)] = panel

combined_weight_grid = pd.DataFrame(rows).sort_values(["cv_utility", "cv_Martin", "dev_total_tc_pct"], ascending=[False, False, True]).reset_index(drop=True)
best_row = combined_weight_grid.loc[0]
best_weights = {"slow": float(best_row["w_slow"]), "medium": float(best_row["w_medium"]), "fast": float(best_row["w_fast"])}

combined_calib_signal_full = combine_deficit_signals(layer_signals_full, best_weights)
combined_calib_panel_full = compute_overlay_panel_from_exposure(base_r_full, combined_calib_signal_full, basket_cost_full)

print("\nCombined-rule candidates")
display(pd.DataFrame([
    {"rule": "min", "weights": "min(slow, medium, fast)"},
    {"rule": "fixed_weighted_deficit", "weights": str(COMBINED_FIXED_WEIGHTS)},
    {"rule": "calibrated_weighted_deficit", "weights": str(best_weights)},
]).set_index("rule"))

print("\nCombined calibrated weight grid (top 12 on dev)")
display(combined_weight_grid.head(12).round(4))

# ------------------------------------------------------------
# Slice OOS panels
# ------------------------------------------------------------
oos_idx = base_r_full.index.intersection(oos.index)

slow_selected_panel = slow_selected_panel_full.loc[slow_selected_panel_full.index.intersection(oos_idx)].copy()
medium_har_panel = medium_har_panel_full.loc[medium_har_panel_full.index.intersection(oos_idx)].copy()
fast_main_panel = fast_main_panel_full.loc[fast_main_panel_full.index.intersection(oos_idx)].copy()

combined_fixed_panel = combined_fixed_panel_full.loc[combined_fixed_panel_full.index.intersection(oos_idx)].copy()
combined_calib_panel = combined_calib_panel_full.loc[combined_calib_panel_full.index.intersection(oos_idx)].copy()
combined_min_panel = combined_min_panel_full.loc[combined_min_panel_full.index.intersection(oos_idx)].copy()

print(
    f"\nCombined calibrated overlay: {int((combined_calib_panel['overlay_turnover'] > 0).sum())} rebalance events | "
    f"total incremental TC (CS): {100.0 * combined_calib_panel['overlay_cost_CS'].sum():.3f}%"
)

def _turnover_weighted_cost_bps(panel: pd.DataFrame) -> float:
    total_turnover = float(panel["overlay_turnover"].sum())
    if total_turnover <= 0:
        return np.nan
    return float(1e4 * panel["overlay_cost_CS"].sum() / total_turnover)


underlying_avg_basket_cost_bps = float(1e4 * basket_cost_full.loc[oos_idx].mean())
underlying_p95_basket_cost_bps = float(1e4 * basket_cost_full.loc[oos_idx].quantile(0.95))

cost_diag_layers = pd.DataFrame({
    "turnover_weighted_cost_bps": {
        slow_selected_label: _turnover_weighted_cost_bps(slow_selected_panel),
        "Medium_HAR": _turnover_weighted_cost_bps(medium_har_panel),
        fast_main_label: _turnover_weighted_cost_bps(fast_main_panel),
        "Combined_fixed": _turnover_weighted_cost_bps(combined_fixed_panel),
        "Combined_calib": _turnover_weighted_cost_bps(combined_calib_panel),
        "Combined_min": _turnover_weighted_cost_bps(combined_min_panel),
    },
    "avg_overlay_turnover": {
        slow_selected_label: float(slow_selected_panel["overlay_turnover"].mean()),
        "Medium_HAR": float(medium_har_panel["overlay_turnover"].mean()),
        fast_main_label: float(fast_main_panel["overlay_turnover"].mean()),
        "Combined_fixed": float(combined_fixed_panel["overlay_turnover"].mean()),
        "Combined_calib": float(combined_calib_panel["overlay_turnover"].mean()),
        "Combined_min": float(combined_min_panel["overlay_turnover"].mean()),
    },
    "total_overlay_cost_%": {
        slow_selected_label: float(100.0 * slow_selected_panel["overlay_cost_CS"].sum()),
        "Medium_HAR": float(100.0 * medium_har_panel["overlay_cost_CS"].sum()),
        fast_main_label: float(100.0 * fast_main_panel["overlay_cost_CS"].sum()),
        "Combined_fixed": float(100.0 * combined_fixed_panel["overlay_cost_CS"].sum()),
        "Combined_calib": float(100.0 * combined_calib_panel["overlay_cost_CS"].sum()),
        "Combined_min": float(100.0 * combined_min_panel["overlay_cost_CS"].sum()),
    },
    "days_with_overlay_trade": {
        slow_selected_label: int((slow_selected_panel["overlay_turnover"] > 0).sum()),
        "Medium_HAR": int((medium_har_panel["overlay_turnover"] > 0).sum()),
        fast_main_label: int((fast_main_panel["overlay_turnover"] > 0).sum()),
        "Combined_fixed": int((combined_fixed_panel["overlay_turnover"] > 0).sum()),
        "Combined_calib": int((combined_calib_panel["overlay_turnover"] > 0).sum()),
        "Combined_min": int((combined_min_panel["overlay_turnover"] > 0).sum()),
    },
}).sort_index()

print("\nOverlay incremental cost diagnostics by layer")
print(
    f"Underlying PC1 one-way basket cost used by all overlay-cost calculations: "
    f"mean={underlying_avg_basket_cost_bps:.2f} bps, p95={underlying_p95_basket_cost_bps:.2f} bps."
)
print("Layer costs differ by exposure turnover and trade timing.")
display(cost_diag_layers.round(4))


# ------------------------------------------------------------
# Benchmarks and final performance table
# ------------------------------------------------------------
qqq_ret = linear_returns_from_prices(
    yf.download(
        "QQQ",
        start=str(base_r_full.index.min().date()),
        end=str((base_r_full.index.max() + pd.Timedelta(days=3)).date()),
        auto_adjust=True,
        progress=False,
    )["Close"]
).squeeze()
qqq_ret = pd.Series(qqq_ret).rename("QQQ").reindex(base_r_full.index).dropna()
qqq_ret_oos = qqq_ret.loc[qqq_ret.index.intersection(oos_idx)].copy()

ew_universe_full = R.loc[base_r_full.index, px_stocks.columns].mean(axis=1).rename("EW_universe_gross")
ew_universe_oos = ew_universe_full.loc[ew_universe_full.index.intersection(oos_idx)].copy()

aligned_inputs = {
    "PC1_base": base_r_full.loc[oos_idx],
    f"{slow_selected_label}_gross": slow_selected_panel["overlay_r_gross"],
    f"{slow_selected_label}_net_CS": slow_selected_panel["overlay_r_net_CS"],
    "Medium_HAR_gross": medium_har_panel["overlay_r_gross"],
    "Medium_HAR_net_CS": medium_har_panel["overlay_r_net_CS"],
    f"{fast_main_label}_gross": fast_main_panel["overlay_r_gross"],
    f"{fast_main_label}_net_CS": fast_main_panel["overlay_r_net_CS"],
    "Combined_fixed_net_CS": combined_fixed_panel["overlay_r_net_CS"],
    "Combined_calib_net_CS": combined_calib_panel["overlay_r_net_CS"],
    "Combined_min_net_CS": combined_min_panel["overlay_r_net_CS"],
    "VolTarget15_net_CS": vol_target_15_panel_full.loc[oos_idx, "overlay_r_net_CS"],
    "VolTarget20_net_CS": vol_target_20_panel_full.loc[oos_idx, "overlay_r_net_CS"],
    "DrawdownControl_net_CS": dd_control_panel_full.loc[oos_idx, "overlay_r_net_CS"],
    "QQQ": qqq_ret_oos,
    "EW_universe_gross": ew_universe_oos,
}

cir_challenger_panel_full_std = globals().get("cir_challenger_panel_full_std", None)

if isinstance(cir_challenger_panel_full_std, pd.DataFrame) and "overlay_r_net_CS" in cir_challenger_panel_full_std.columns:
    cir_oos_panel = cir_challenger_panel_full_std.loc[
        cir_challenger_panel_full_std.index.intersection(oos_idx)
    ].copy()
    aligned_inputs["CIR_challenger_net_CS"] = cir_oos_panel["overlay_r_net_CS"]
    print("Added CIR_challenger_net_CS to final aligned table.")
else:
    print("CIR challenger not available or not in standard panel format — skipping CIR in final aligned table.")

aligned = pd.concat(aligned_inputs, axis=1).dropna()

print("\n=== Full performance comparison, aligned dates ===")
print(f"Aligned sample: {aligned.index.min().date()} -> {aligned.index.max().date()} | n={len(aligned)}")
perf_final = perf_full({c: aligned[c] for c in aligned.columns})
display(perf_final.round(4))

# ------------------------------------------------------------
# Subperiod analysis
# ------------------------------------------------------------
subperiods = {
    "2018-2019": ("2018-01-01", "2019-12-31"),
    "2020": ("2020-01-01", "2020-12-31"),
    "2021": ("2021-01-01", "2021-12-31"),
    "2022": ("2022-01-01", "2022-12-31"),
    "2023-2026": ("2023-01-01", str(aligned.index.max().date())),
}

subperiod_rows = []
key_strategies = [
    "PC1_base",
    f"{slow_selected_label}_net_CS",
    "Medium_HAR_net_CS",
    f"{fast_main_label}_net_CS",
    "Combined_fixed_net_CS",
    "Combined_calib_net_CS",
    "Combined_min_net_CS",
    "VolTarget15_net_CS",
    "VolTarget20_net_CS",
    "DrawdownControl_net_CS",
    "QQQ",
]
if "CIR_challenger_net_CS" in aligned.columns:
    key_strategies.insert(3, "CIR_challenger_net_CS")

for label, (d0, d1) in subperiods.items():
    sub = aligned.loc[d0:d1, key_strategies].dropna(how="all")
    if sub.empty:
        continue
    stats = perf_full({c: sub[c].dropna() for c in sub.columns})
    stats["subperiod"] = label
    subperiod_rows.append(stats.reset_index())

subperiod_perf = pd.concat(subperiod_rows, axis=0, ignore_index=True) if subperiod_rows else pd.DataFrame()
if not subperiod_perf.empty:
    print("\nSubperiod analysis (selected net strategies)")
    display(subperiod_perf.set_index(["subperiod", "strategy"]).round(4))

# ------------------------------------------------------------
# Bootstrap inference
# ------------------------------------------------------------
boot_slow, boot_slow_summary = bootstrap_compare(
    base=aligned["PC1_base"],
    target=aligned[f"{slow_selected_label}_net_CS"],
    label=f"{slow_selected_label}_net_CS vs base",
    n_boot=BOOT_B_FINAL,
    block_len=BLOCK_LEN,
    seed=42,
)

boot_combined_fixed, boot_combined_fixed_summary = bootstrap_compare(
    base=aligned["PC1_base"],
    target=aligned["Combined_fixed_net_CS"],
    label="Combined_fixed_net_CS vs base",
    n_boot=BOOT_B_FINAL,
    block_len=BLOCK_LEN,
    seed=123,
)

boot_combined_calib, boot_combined_calib_summary = bootstrap_compare(
    base=aligned["PC1_base"],
    target=aligned["Combined_calib_net_CS"],
    label="Combined_calib_net_CS vs base",
    n_boot=BOOT_B_FINAL,
    block_len=BLOCK_LEN,
    seed=321,
)

boot_fast, boot_fast_summary = bootstrap_compare(
    base=aligned["PC1_base"],
    target=aligned[f"{fast_main_label}_net_CS"],
    label=f"{fast_main_label}_net_CS vs base",
    n_boot=BOOT_B_FINAL,
    block_len=BLOCK_LEN,
    seed=777,
)

# ------------------------------------------------------------
# Diagnostic plots
# ------------------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

pd.DataFrame({
    "PC1_base": (1.0 + aligned["PC1_base"]).cumprod(),
    f"{slow_selected_label}_net_CS": (1.0 + aligned[f"{slow_selected_label}_net_CS"]).cumprod(),
    "Combined_fixed_net_CS": (1.0 + aligned["Combined_fixed_net_CS"]).cumprod(),
    "Combined_calib_net_CS": (1.0 + aligned["Combined_calib_net_CS"]).cumprod(),
    "VolTarget20_net_CS": (1.0 + aligned["VolTarget20_net_CS"]).cumprod(),
    "QQQ": (1.0 + aligned["QQQ"]).cumprod(),
}).plot(ax=axes[0], title="Equity curves — base vs selected slow / combined / benchmark")
axes[0].set_ylabel("Cumulative wealth")

pd.DataFrame({
    "slow_exp": slow_selected_panel["exposure_applied"],
    "medium_exp": medium_har_panel["exposure_applied"],
    "fast_exp": fast_main_panel["exposure_applied"],
    "combined_fixed_exp": combined_fixed_panel["exposure_applied"],
    "combined_calib_exp": combined_calib_panel["exposure_applied"],
}).plot(ax=axes[1], title="Layer exposures (OOS)")
axes[1].set_ylabel("Exposure")

dd_df = pd.DataFrame({
    "PC1_base": (1.0 + aligned["PC1_base"]).cumprod() / (1.0 + aligned["PC1_base"]).cumprod().cummax() - 1.0,
    f"{slow_selected_label}_net_CS": (1.0 + aligned[f"{slow_selected_label}_net_CS"]).cumprod() / (1.0 + aligned[f"{slow_selected_label}_net_CS"]).cumprod().cummax() - 1.0,
    "Combined_fixed_net_CS": (1.0 + aligned["Combined_fixed_net_CS"]).cumprod() / (1.0 + aligned["Combined_fixed_net_CS"]).cumprod().cummax() - 1.0,
    "Combined_calib_net_CS": (1.0 + aligned["Combined_calib_net_CS"]).cumprod() / (1.0 + aligned["Combined_calib_net_CS"]).cumprod().cummax() - 1.0,
    "VolTarget20_net_CS": (1.0 + aligned["VolTarget20_net_CS"]).cumprod() / (1.0 + aligned["VolTarget20_net_CS"]).cumprod().cummax() - 1.0,
    "QQQ": (1.0 + aligned["QQQ"]).cumprod() / (1.0 + aligned["QQQ"]).cumprod().cummax() - 1.0,
})
dd_df.plot(ax=axes[2], title="Drawdown — selected slow / combined / benchmark")
axes[2].set_ylabel("Drawdown")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Save objects
# ------------------------------------------------------------
combined_fixed_main = combined_fixed_panel.copy()
combined_calib_main = combined_calib_panel.copy()
combined_min_benchmark = combined_min_panel.copy()
selected_slow_main = slow_selected_panel.copy()
fast_main_selected = fast_main_panel.copy()
deterministic_benchmarks = {
    "VolTarget15_net_CS": vol_target_15_panel_full.loc[oos_idx].copy(),
    "VolTarget20_net_CS": vol_target_20_panel_full.loc[oos_idx].copy(),
    "DrawdownControl_net_CS": dd_control_panel_full.loc[oos_idx].copy(),
}

print("\nSaved: selected_slow_main, fast_main_selected, combined_fixed_main, combined_calib_main, combined_min_benchmark, perf_final, subperiod_perf, deterministic_benchmarks")


## Notes for next iterations

- The current retained result is the standalone slow OU overlay.
- HAR-RV, CIR and Kalman/Mahalanobis layers remain challengers/extensions.
- The calibrated combined rule currently assigns zero weight to the fast layer; this is useful evidence, not a reason to delete the fast research path.
- Before a journal submission, the most valuable extension is a walk-forward or regime-conditioned combination policy with ablation tests across markets/universes.
